# Method 3: supervised pairwise GBDT + retrieval blocking + hard negatives

Self-contained: this notebook writes its own source files, installs what Colab lacks and runs the full pipeline. Set `DATA_DIR` in the next cell, then **Runtime → Run all**.

`DATA_DIR` must contain `train/` (`train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, `train_ground_truth.tsv`) and `test/` (`test_source1.tsv`, `test_source2.tsv`, `test_source3.tsv`).

In [ ]:
# ===================== settings =====================
DATA_DIR = "/content/dataset"          # folder that contains train/ and test/
OUT_DIR = "/content/outputs/method3_gbdt"   # results are written here
MOUNT_GOOGLE_DRIVE = False              # True if DATA_DIR / OUT_DIR are on Drive (/content/drive/MyDrive/...)
COMMAND = "all"                         # "validate" (CV on train), "predict" (test submission) or "all"
N_FOLDS = 5                             # cross-validation folds for "validate"

In [ ]:
import os

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

for split in ("train", "test"):
    path = os.path.join(DATA_DIR, split)
    assert os.path.isdir(path), f"missing folder: {path}"
    print(path, sorted(os.listdir(path)))

In [ ]:
%pip install -q rapidfuzz==3.13.0 lightgbm==4.6.0 tqdm

## Source code

The cells below write the pipeline into `er_src/`. They mirror the repository files.

In [ ]:
import os

os.makedirs("er_src/er_common", exist_ok=True)
os.makedirs("er_src/method2_tfidf_retrieval", exist_ok=True)
os.makedirs("er_src/method3_gbdt", exist_ok=True)
open("er_src/er_common/__init__.py", "w").close()

In [ ]:
%%writefile er_src/er_common/progress.py
"""Progress bars (tqdm) shared by every stage.

``progress`` wraps an iterable, ``stages`` gives a bar that ticks once per named step (for
stretches of vectorised work that have no natural loop). Bars go to stderr and cooperate with
logging (see ``runner.main``); ``set_enabled(False)`` / ``--no-progress`` turns them off.
"""

from __future__ import annotations

from collections.abc import Iterable, Iterator
from contextlib import contextmanager
from typing import Any, TypeVar

from tqdm.auto import tqdm

T = TypeVar("T")

_ENABLED = True


def set_enabled(enabled: bool) -> None:
    global _ENABLED
    _ENABLED = enabled


def _kw(desc: str, total: int | None, unit: str) -> dict[str, Any]:
    return {
        "desc": desc,
        "total": total,
        "unit": unit,
        "disable": not _ENABLED,
        "dynamic_ncols": True,
        "mininterval": 0.5,
        "leave": False,
        "unit_scale": True,
    }


def progress(it: Iterable[T], desc: str, total: int | None = None, unit: str = "it") -> Iterator[T]:
    """Iterate ``it`` with a progress bar."""
    yield from tqdm(it, **_kw(desc, total, unit))


def bar(desc: str, total: int, unit: str = "it") -> tqdm:
    """Manually updated bar (``with bar(...) as pb: pb.update(n)``)."""
    return tqdm(**_kw(desc, total, unit))


class _Stages:
    def __init__(self, pb: tqdm) -> None:
        self.pb = pb

    def __call__(self, name: str) -> None:
        """Mark the previous step done and show ``name`` as the current one."""
        if self.pb.n or self.pb.postfix:
            self.pb.update(1)
        self.pb.set_postfix_str(name, refresh=True)


@contextmanager
def stages(desc: str, total: int) -> Iterator[_Stages]:
    """Bar that advances once per named step: ``with stages("features", 5) as step: step("names") ...``."""
    with tqdm(**{**_kw(desc, total, "step"), "unit_scale": False}) as pb:
        s = _Stages(pb)
        yield s
        pb.update(pb.total - pb.n)

In [ ]:
%%writefile er_src/er_common/io.py
"""Data loading, output writing and submission-format validation.

All challenge files are TSV. Two traps handled here:
  * pandas turns strings such as "NA", "None", "null" into NaN by default -> a business
    literally named "NA" would silently vanish. We read everything as ``str`` with
    ``keep_default_na=False``.
  * Business names can contain double quotes. The organisers' snippet uses pandas' default
    (QUOTE_MINIMAL) parsing, so we use it too, but verify the parsed row count against the raw
    line count and fall back to ``QUOTE_NONE`` if a stray quote swallowed lines.
"""

from __future__ import annotations

import csv
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from er_common.progress import stages

RECORD_COLUMNS = ("entity_id", "business_name", "business_address", "country")
GT_COLUMNS = ("source1_entity_id", "matched_entity_ids")
MATCH_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


@dataclass(frozen=True)
class SplitData:
    """One split (train or test). ``s23`` is S2 and S3 stacked, with a ``source`` column."""

    split: str
    s1: pd.DataFrame
    s23: pd.DataFrame
    gt: dict[str, tuple[str, ...]] | None

    @property
    def s1_ids(self) -> list[str]:
        return self.s1["entity_id"].tolist()


def _count_data_lines(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as fh:
        n = sum(1 for line in fh if line.strip("\r\n"))
    return max(n - 1, 0)


def read_tsv(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    expected = _count_data_lines(path)
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8")
    if len(df) != expected:
        df = pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            na_filter=False,
            quoting=csv.QUOTE_NONE,
            encoding="utf-8",
        )
    if len(df) != expected:
        raise ValueError(f"{path}: parsed {len(df)} rows but file has {expected} data lines")
    return df.rename(columns=lambda c: str(c).strip())


def _load_records(path: Path, prefix: str) -> pd.DataFrame:
    df = read_tsv(path)
    missing = [c for c in RECORD_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}; got {list(df.columns)}")
    df = df.loc[:, list(RECORD_COLUMNS)].copy()
    for col in RECORD_COLUMNS:
        df[col] = df[col].astype(str).str.strip()
    if df["entity_id"].duplicated().any():
        dups = df.loc[df["entity_id"].duplicated(), "entity_id"].head(5).tolist()
        raise ValueError(f"{path}: duplicate entity_id values, e.g. {dups}")
    bad = df.loc[~df["entity_id"].str.startswith(prefix), "entity_id"].head(5).tolist()
    if bad:
        raise ValueError(f"{path}: entity_id without expected prefix {prefix!r}: {bad}")
    return df.reset_index(drop=True)


def parse_id_list(value: str) -> tuple[str, ...]:
    """Comma-separated ID list -> de-duplicated tuple preserving order."""
    out: list[str] = []
    seen: set[str] = set()
    for tok in value.split(","):
        tok = tok.strip()
        if tok and tok not in seen:
            seen.add(tok)
            out.append(tok)
    return tuple(out)


def load_ground_truth(path: str | Path) -> dict[str, tuple[str, ...]]:
    df = read_tsv(path)
    missing = [c for c in GT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}")
    gt: dict[str, tuple[str, ...]] = {}
    for s1_id, ids in zip(df["source1_entity_id"], df["matched_entity_ids"], strict=True):
        gt[s1_id.strip()] = parse_id_list(ids)
    return gt


def load_split(data_dir: str | Path, split: str) -> SplitData:
    """Load ``<data_dir>/<split>/<split>_source{1,2,3}.tsv`` (+ ground truth for train)."""
    base = Path(data_dir) / split
    with stages(f"load {split}", 4) as step:
        step(f"{split}_source1.tsv")
        s1 = _load_records(base / f"{split}_source1.tsv", "S1-")
        step(f"{split}_source2.tsv")
        s2 = _load_records(base / f"{split}_source2.tsv", "S2-")
        step(f"{split}_source3.tsv")
        s3 = _load_records(base / f"{split}_source3.tsv", "S3-")
        step("ground truth")
        gt_path = base / f"{split}_ground_truth.tsv"
        gt = load_ground_truth(gt_path) if gt_path.is_file() else None
    s2["source"] = "S2"
    s3["source"] = "S3"
    s23 = pd.concat([s2, s3], ignore_index=True)
    s1["source"] = "S1"

    if gt is not None:
        _check_gt(gt, s1, s23, gt_path)
    return SplitData(split=split, s1=s1, s23=s23, gt=gt)


def _check_gt(gt: Mapping[str, Sequence[str]], s1: pd.DataFrame, s23: pd.DataFrame, path: Path) -> None:
    s1_ids = set(s1["entity_id"])
    s23_ids = set(s23["entity_id"])
    unknown_s1 = [k for k in gt if k not in s1_ids]
    unknown_c = sorted({c for v in gt.values() for c in v if c not in s23_ids})
    if unknown_s1 or unknown_c:
        raise ValueError(f"{path}: ground truth references unknown ids: s1={unknown_s1[:5]} s2/s3={unknown_c[:5]}")


def gt_for(gt: Mapping[str, Sequence[str]], s1_ids: Iterable[str]) -> dict[str, tuple[str, ...]]:
    """S1 entities absent from the GT file are singletons (empty match set)."""
    return {s: tuple(gt.get(s, ())) for s in s1_ids}


def candidate_owner_multiplicity(gt: Mapping[str, Sequence[str]]) -> int:
    """Max number of S1 entities a single S2/S3 record is matched to in the GT.

    1 means every S2/S3 record belongs to at most one S1 entity -> the exclusivity constraint
    used by the decision layer is consistent with the labels.
    """
    counts: dict[str, int] = {}
    for ids in gt.values():
        for c in ids:
            counts[c] = counts.get(c, 0) + 1
    return max(counts.values(), default=0)


def write_id_lists(
    path: str | Path, header: Sequence[str], rows: Mapping[str, Sequence[str]], order: Sequence[str]
) -> None:
    """Write ``id<TAB>comma,list`` rows in ``order``; entities without a list get an empty cell."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as fh:
        fh.write("\t".join(header) + "\n")
        for s1_id in order:
            ids = rows.get(s1_id, ())
            fh.write(f"{s1_id}\t{','.join(ids)}\n")


def validate_submission(
    matches: Mapping[str, Sequence[str]],
    candidates: Mapping[str, Sequence[str]],
    s1_ids: Sequence[str],
    s23_ids: Iterable[str],
) -> list[str]:
    """Return a list of rule violations (empty list == valid)."""
    errors: list[str] = []
    valid_c = set(s23_ids)
    s1_set = set(s1_ids)
    if len(s1_set) != len(s1_ids):
        errors.append("duplicate Source 1 ids in output order")
    for name, table in (("matching_results", matches), ("candidate_pairs", candidates)):
        extra = set(table) - s1_set
        if extra:
            errors.append(f"{name}: rows for unknown S1 ids, e.g. {sorted(extra)[:3]}")
        for s1_id, ids in table.items():
            if len(set(ids)) != len(ids):
                errors.append(f"{name}: duplicate ids for {s1_id}")
            bad = [c for c in ids if c not in valid_c or not c.startswith(("S2-", "S3-"))]
            if bad:
                errors.append(f"{name}: invalid ids for {s1_id}: {bad[:3]}")
    for s1_id, ids in matches.items():
        missing = set(ids) - set(candidates.get(s1_id, ()))
        if missing:
            errors.append(f"match not in candidate list for {s1_id}: {sorted(missing)[:3]}")
    return errors


def write_submission(
    out_dir: str | Path,
    matches: Mapping[str, Sequence[str]],
    candidates: Mapping[str, Sequence[str]],
    s1_ids: Sequence[str],
    s23_ids: Iterable[str],
) -> None:
    errors = validate_submission(matches, candidates, s1_ids, s23_ids)
    if errors:
        raise ValueError("submission violates challenge constraints:\n  " + "\n  ".join(errors[:20]))
    out = Path(out_dir)
    write_id_lists(out / "matching_results.tsv", MATCH_HEADER, matches, s1_ids)
    write_id_lists(out / "candidate_pairs.tsv", CANDIDATE_HEADER, candidates, s1_ids)

In [ ]:
%%writefile er_src/er_common/normalize.py
"""Field normalisation shared by every method.

Design rules
  * Canonicalise *both* sides of a pair to the same token (``street``/``st``/``str`` -> ``st``);
    whether ``st`` meant Street or Saint is irrelevant as long as both sides agree.
  * Never throw identity information away: numbers, postal codes and landmarks are extracted
    into their own fields instead of being deleted.
  * Country is an open set: known aliases collapse (``USA`` -> ``us``) and anything unseen
    passes through lower-cased, so France (test-only) is handled like any other label.
  * The abbreviation tables are generic linguistic normalisation (no business lookup).
"""

from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass
from functools import cache

import pandas as pd

from er_common.progress import progress

# --------------------------------------------------------------------------------------------
# Lexicons
# --------------------------------------------------------------------------------------------

_SHARED_MAP: dict[str, str] = {
    "saint": "st",
    "sainte": "ste",
    "mount": "mt",
    "fort": "ft",
    "sri": "sri",
    "shri": "sri",
    "shree": "sri",
    "sree": "sri",
    "shrii": "sri",
    "shreee": "sri",
    "smt": "smt",
    "shrimati": "smt",
    "srimati": "smt",
    "mohd": "mohd",
    "mohammed": "mohd",
    "mohammad": "mohd",
    "muhammad": "mohd",
    "mohamed": "mohd",
    "muhammed": "mohd",
    "mohamad": "mohd",
    "mahmood": "mahmud",
    "mahmud": "mahmud",
    "and": "and",
}

_NAME_MAP: dict[str, str] = {
    **_SHARED_MAP,
    "private": "pvt",
    "pvt": "pvt",
    "pte": "pvt",
    "limited": "ltd",
    "ltd": "ltd",
    "ltda": "ltd",
    "corporation": "corp",
    "corpn": "corp",
    "corp": "corp",
    "incorporated": "inc",
    "inc": "inc",
    "company": "co",
    "companies": "co",
    "cos": "co",
    "compagnie": "co",
    "cie": "co",
    "international": "intl",
    "intl": "intl",
    "internatl": "intl",
    "national": "natl",
    "natl": "natl",
    "manufacturing": "mfg",
    "manufacturers": "mfg",
    "manufacturer": "mfg",
    "mfrs": "mfg",
    "mfg": "mfg",
    "services": "svc",
    "service": "svc",
    "svcs": "svc",
    "svc": "svc",
    "serv": "svc",
    "associates": "assoc",
    "association": "assoc",
    "assn": "assoc",
    "assoc": "assoc",
    "brothers": "bros",
    "bros": "bros",
    "freres": "bros",
    "enterprises": "ent",
    "enterprise": "ent",
    "entp": "ent",
    "ent": "ent",
    "hospital": "hosp",
    "hospitals": "hosp",
    "hosp": "hosp",
    "center": "ctr",
    "centre": "ctr",
    "cntr": "ctr",
    "ctr": "ctr",
    "department": "dept",
    "dept": "dept",
    "university": "univ",
    "univ": "univ",
    "institute": "inst",
    "inst": "inst",
    "technologies": "tech",
    "technology": "tech",
    "techs": "tech",
    "tech": "tech",
    "systems": "sys",
    "system": "sys",
    "sys": "sys",
    "management": "mgmt",
    "mgmt": "mgmt",
    "development": "dev",
    "dev": "dev",
    "industries": "ind",
    "industry": "ind",
    "inds": "ind",
    "ind": "ind",
    "market": "mkt",
    "markets": "mkt",
    "mkt": "mkt",
    "doctor": "dr",
    "dr": "dr",
    "pharmaceuticals": "pharma",
    "pharmaceutical": "pharma",
    "pharma": "pharma",
    "medical": "med",
    "med": "med",
    "group": "grp",
    "grp": "grp",
    "holdings": "hldg",
    "hldgs": "hldg",
    "hldg": "hldg",
    "financial": "fin",
    "finance": "fin",
    "fin": "fin",
    "solutions": "soln",
    "solns": "soln",
    "soln": "soln",
    "consultants": "consult",
    "consultancy": "consult",
    "consulting": "consult",
    "consult": "consult",
    "engineering": "eng",
    "engg": "eng",
    "eng": "eng",
    "electricals": "elec",
    "electrical": "elec",
    "electric": "elec",
    "elec": "elec",
    "travels": "travel",
    "travel": "travel",
    "traders": "trade",
    "trader": "trade",
    "trading": "trade",
    "automobiles": "auto",
    "automobile": "auto",
    "automotive": "auto",
    "auto": "auto",
    "motors": "motor",
    "motor": "motor",
    "laboratories": "lab",
    "laboratory": "lab",
    "labs": "lab",
    "lab": "lab",
    "clinics": "clinic",
    "clinic": "clinic",
    "agencies": "agency",
    "agency": "agency",
    "insurance": "ins",
    "ins": "ins",
    "communications": "comm",
    "communication": "comm",
    "comm": "comm",
    "products": "prod",
    "product": "prod",
    "prod": "prod",
    "restaurants": "restaurant",
    "hotels": "hotel",
    "stores": "store",
    "shoppe": "shop",
    "etablissements": "ets",
    "etablissement": "ets",
    "ets": "ets",
    "societe": "ste",
    "ste": "ste",
}

_ADDR_MAP: dict[str, str] = {
    **_SHARED_MAP,
    "road": "rd",
    "rd": "rd",
    "street": "st",
    "st": "st",
    "str": "st",
    "stt": "st",
    "avenue": "ave",
    "ave": "ave",
    "av": "ave",
    "avn": "ave",
    "boulevard": "blvd",
    "blvd": "blvd",
    "bd": "blvd",
    "boul": "blvd",
    "bld": "blvd",
    "lane": "ln",
    "ln": "ln",
    "drive": "dr",
    "dr": "dr",
    "drv": "dr",
    "court": "ct",
    "ct": "ct",
    "place": "pl",
    "pl": "pl",
    "square": "sq",
    "sq": "sq",
    "highway": "hwy",
    "hwy": "hwy",
    "hiway": "hwy",
    "parkway": "pkwy",
    "pkwy": "pkwy",
    "expressway": "expy",
    "expy": "expy",
    "terrace": "ter",
    "terr": "ter",
    "circle": "cir",
    "cir": "cir",
    "crescent": "cres",
    "cres": "cres",
    "plaza": "plz",
    "plz": "plz",
    "suite": "ste",
    "suit": "ste",
    "apartment": "apt",
    "apt": "apt",
    "floor": "fl",
    "flr": "fl",
    "fl": "fl",
    "building": "bldg",
    "bldg": "bldg",
    "bldng": "bldg",
    "room": "rm",
    "rm": "rm",
    "north": "n",
    "south": "s",
    "east": "e",
    "west": "w",
    "northeast": "ne",
    "northwest": "nw",
    "southeast": "se",
    "southwest": "sw",
    "nagar": "nagar",
    "ngr": "nagar",
    "nagr": "nagar",
    "marg": "marg",
    "mrg": "marg",
    "colony": "colony",
    "clny": "colony",
    "col": "colony",
    "sector": "sec",
    "sect": "sec",
    "sec": "sec",
    "phase": "phase",
    "ph": "phase",
    "block": "blk",
    "blk": "blk",
    "cross": "cross",
    "crs": "cross",
    "layout": "layout",
    "lyt": "layout",
    "extension": "extn",
    "extn": "extn",
    "ext": "extn",
    "bazaar": "bazar",
    "bazar": "bazar",
    "bzr": "bazar",
    "chowk": "chowk",
    "chk": "chowk",
    "mohalla": "mohalla",
    "mohala": "mohalla",
    "enclave": "enclave",
    "encl": "enclave",
    "estate": "estate",
    "est": "estate",
    "industrial": "indl",
    "indl": "indl",
    "plot": "plot",
    "plt": "plot",
    "opposite": "opp",
    "opp": "opp",
    "near": "near",
    "nr": "near",
    "behind": "behind",
    "bhd": "behind",
    "junction": "jn",
    "jn": "jn",
    "jct": "jn",
    "junc": "jn",
    "station": "stn",
    "stn": "stn",
    "railway": "rly",
    "rly": "rly",
    "district": "dist",
    "distt": "dist",
    "dist": "dist",
    "village": "vill",
    "vill": "vill",
    "vil": "vill",
    "taluka": "taluk",
    "taluk": "taluk",
    "chemin": "chemin",
    "ch": "chemin",
    "impasse": "imp",
    "imp": "imp",
    "route": "rte",
    "rte": "rte",
    "faubourg": "fbg",
    "fbg": "fbg",
    "first": "1",
    "second": "2",
    "third": "3",
    "fourth": "4",
    "fifth": "5",
    "sixth": "6",
    "seventh": "7",
    "eighth": "8",
    "ninth": "9",
    "tenth": "10",
    # renamed cities: both spellings are live in real address data
    "bombay": "mumbai",
    "madras": "chennai",
    "calcutta": "kolkata",
    "bangalore": "bengaluru",
    "gurgaon": "gurugram",
    "poona": "pune",
    "trivandrum": "thiruvananthapuram",
    "baroda": "vadodara",
    "benares": "varanasi",
    "banaras": "varanasi",
    "benaras": "varanasi",
    "cochin": "kochi",
    "mysore": "mysuru",
    "mangalore": "mangaluru",
    "allahabad": "prayagraj",
    "pondicherry": "puducherry",
    "simla": "shimla",
    "calicut": "kozhikode",
    "belgaum": "belagavi",
    "vizag": "visakhapatnam",
    "nyc": "ny",
    # full state names -> postal codes (codes themselves are left untouched)
    "alabama": "al",
    "alaska": "ak",
    "arizona": "az",
    "arkansas": "ar",
    "california": "ca",
    "colorado": "co",
    "connecticut": "ct",
    "delaware": "de",
    "florida": "fl",
    "georgia": "ga",
    "hawaii": "hi",
    "idaho": "id",
    "illinois": "il",
    "indiana": "in",
    "iowa": "ia",
    "kansas": "ks",
    "kentucky": "ky",
    "louisiana": "la",
    "maine": "me",
    "maryland": "md",
    "massachusetts": "ma",
    "michigan": "mi",
    "minnesota": "mn",
    "mississippi": "ms",
    "missouri": "mo",
    "montana": "mt",
    "nebraska": "ne",
    "nevada": "nv",
    "ohio": "oh",
    "oklahoma": "ok",
    "oregon": "or",
    "pennsylvania": "pa",
    "tennessee": "tn",
    "texas": "tx",
    "utah": "ut",
    "vermont": "vt",
    "virginia": "va",
    "washington": "wa",
    "wisconsin": "wi",
    "wyoming": "wy",
    "maharashtra": "mh",
    "karnataka": "ka",
    "kerala": "kl",
    "gujarat": "gj",
    "rajasthan": "rj",
    "bihar": "br",
    "odisha": "od",
    "orissa": "od",
    "telangana": "ts",
    "punjab": "pb",
    "haryana": "hr",
    "jharkhand": "jh",
    "chhattisgarh": "cg",
    "uttarakhand": "uk",
    "assam": "as",
    "goa": "ga",
}

# Multi-word phrases rewritten before token mapping (regex, replacement).
_ADDR_PHRASES: tuple[tuple[str, str], ...] = (
    ("mahatma gandhi", "mg"),
    ("new york", "ny"),
    ("new jersey", "nj"),
    ("new mexico", "nm"),
    ("new hampshire", "nh"),
    ("north carolina", "nc"),
    ("south carolina", "sc"),
    ("north dakota", "nd"),
    ("south dakota", "sd"),
    ("west virginia", "wv"),
    ("rhode island", "ri"),
    ("district of columbia", "dc"),
    ("uttar pradesh", "up"),
    ("madhya pradesh", "mp"),
    ("andhra pradesh", "ap"),
    ("himachal pradesh", "hp"),
    ("arunachal pradesh", "ar"),
    ("tamil nadu", "tn"),
    ("west bengal", "wb"),
    ("jammu and kashmir", "jk"),
    ("post office", "po"),
    ("p o", "po"),
    ("united states of america", ""),
    ("united states", ""),
)
_ADDR_PHRASE_RES = tuple((re.compile(rf"\b{p}\b"), r) for p, r in _ADDR_PHRASES)

LEGAL_TOKENS: frozenset[str] = frozenset(
    {
        "pvt",
        "ltd",
        "llc",
        "llp",
        "inc",
        "corp",
        "co",
        "plc",
        "lp",
        "pllc",
        "pc",
        "gmbh",
        "sarl",
        "sas",
        "sasu",
        "sa",
        "eurl",
        "snc",
        "sci",
        "ag",
        "bv",
        "nv",
        "opc",
        "lllp",
    }
)
# country words that appear as name tails ("XYZ (India) Pvt Ltd")
_NAME_TAIL_EXTRA: frozenset[str] = frozenset({"india", "usa", "us", "america", "france", "and"})
_NAME_STOP: frozenset[str] = frozenset({"the", "and", "of", "le", "la", "les", "de", "du", "des", "et"})
_ADDR_DROP: frozenset[str] = frozenset({"india", "usa", "france", "cedex"})
_NUMBER_WORDS: frozenset[str] = frozenset({"no", "number", "num", "nos"})

_COUNTRY_ALIASES: dict[str, str] = {
    "us": "us",
    "usa": "us",
    "u s": "us",
    "u s a": "us",
    "united states": "us",
    "united states of america": "us",
    "america": "us",
    "in": "in",
    "ind": "in",
    "india": "in",
    "bharat": "in",
    "republic of india": "in",
    "fr": "fr",
    "fra": "fr",
    "france": "fr",
    "republique francaise": "fr",
    "french republic": "fr",
}

# keyword + at most two following words, never crossing a comma: without separators we cannot
# tell where "Near Clock Tower Mumbai" ends, and eating the city is worse than keeping a word.
_LANDMARK_RE = re.compile(
    r"\b(?:near|nr|opp|opposite|behind|beside|besides|next to|adjacent to|adj to|adj|in front of|"
    r"infront of|close to|facing|across from|across)\b\.?(?:[ \t]+[^\s,;]+){0,2}"
)
_DBA_RE = re.compile(r"\b(?:d b a|dba|doing business as|t a|trading as|aka|a k a)\b")

_APOS_RE = re.compile(r"[’‘'`´]")
_ZIP4_RE = re.compile(r"\b(\d{5})\s*-\s*\d{4}\b")
_PIN_SPLIT_RE = re.compile(r"\b(\d{3})\s(\d{3})\s*$")
_ORDINAL_RE = re.compile(r"\b(\d+)(?:st|nd|rd|th)\b")
_PUNCT_RE = re.compile(r"[^\w\s]|_")
_ALNUM_SPLIT_RE = re.compile(r"(?<=[^\W\d_])(?=\d)|(?<=\d)(?=[^\W\d_])")
_WS_RE = re.compile(r"\s+")
_POSTAL_RE = re.compile(r"^\d{5,6}$")
_DIGITS_RE = re.compile(r"\d+")
_VOWELS = frozenset("aeiou")


# --------------------------------------------------------------------------------------------
# Primitive steps
# --------------------------------------------------------------------------------------------


def fold_unicode(text: str) -> str:
    """NFKD + strip combining marks (é -> e, ç -> c) + lower-case."""
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch)).lower()


def _clean(text: str) -> str:
    text = fold_unicode(text).replace("&", " and ").replace("@", " at ")
    text = _APOS_RE.sub("", text)
    text = _ZIP4_RE.sub(r"\1", text)
    text = _ORDINAL_RE.sub(r"\1", text)
    text = _PUNCT_RE.sub(" ", text)
    text = _ALNUM_SPLIT_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).strip()


def _join_initials(tokens: list[str]) -> list[str]:
    """``s b i`` -> ``sbi`` (initialisms written with dots/spaces)."""
    out: list[str] = []
    run: list[str] = []
    for tok in tokens:
        if len(tok) == 1 and tok.isalpha():
            run.append(tok)
            continue
        if len(run) >= 2:
            out.append("".join(run))
        else:
            out.extend(run)
        run = []
        out.append(tok)
    if len(run) >= 2:
        out.append("".join(run))
    else:
        out.extend(run)
    return out


def _map_tokens(tokens: list[str], mapping: dict[str, str]) -> list[str]:
    return [m for m in (mapping.get(t, t) for t in tokens) if m]


def skeleton(token: str) -> str:
    """Transliteration-tolerant consonant skeleton: ``shree``/``sri`` -> ``sr``, ``baalaji`` -> ``blj``."""
    t = token
    for a, b in (
        ("ph", "f"),
        ("sh", "s"),
        ("kh", "k"),
        ("gh", "g"),
        ("th", "t"),
        ("dh", "d"),
        ("bh", "b"),
        ("jh", "j"),
        ("ch", "c"),
        ("ck", "k"),
        ("q", "k"),
        ("w", "v"),
        ("z", "j"),
        ("x", "ks"),
        ("y", "i"),
    ):
        t = t.replace(a, b)
    if not t:
        return t
    head, rest = t[0], "".join(ch for ch in t[1:] if ch not in _VOWELS)
    out = [head]
    for ch in rest:
        if ch != out[-1]:
            out.append(ch)
    return "".join(out)


def normalize_country(value: str) -> str:
    key = _WS_RE.sub(" ", _PUNCT_RE.sub(" ", fold_unicode(value))).strip()
    return _COUNTRY_ALIASES.get(key, key)


# --------------------------------------------------------------------------------------------
# Names
# --------------------------------------------------------------------------------------------


@dataclass(frozen=True)
class NormName:
    norm: str  # all tokens, canonicalised
    core: str  # legal suffixes / country tail / stop words removed
    alt: str  # trade name after "dba", "" if none
    legal: str  # sorted legal tokens found
    skel: str  # consonant skeleton of core tokens
    acronym: str  # initials of core tokens (>= 2 tokens), "" otherwise


def _strip_tail(tokens: list[str]) -> tuple[list[str], list[str]]:
    legal: list[str] = []
    end = len(tokens)
    while end > 1 and (tokens[end - 1] in LEGAL_TOKENS or tokens[end - 1] in _NAME_TAIL_EXTRA):
        if tokens[end - 1] in LEGAL_TOKENS:
            legal.append(tokens[end - 1])
        end -= 1
    return tokens[:end], legal


def _core_tokens(tokens: list[str]) -> tuple[list[str], list[str]]:
    body, legal = _strip_tail(tokens)
    legal += [t for t in body if t in LEGAL_TOKENS and t not in {"co", "sa", "ag", "pc"}]
    core = [t for t in body if t not in _NAME_STOP and not (t in LEGAL_TOKENS and t != body[0])]
    return (core or body), legal


@cache
def normalize_name(raw: str) -> NormName:
    cleaned = _clean(raw)
    parts = _DBA_RE.split(cleaned, maxsplit=1)
    main = _map_tokens(_join_initials(parts[0].split()), _NAME_MAP)
    alt_tokens = _map_tokens(_join_initials(parts[1].split()), _NAME_MAP) if len(parts) > 1 else []
    core, legal = _core_tokens(main) if main else ([], [])
    alt_core = _core_tokens(alt_tokens)[0] if alt_tokens else []
    acronym = "".join(t[0] for t in core if t) if len(core) >= 2 else ""
    return NormName(
        norm=" ".join(main + alt_tokens),
        core=" ".join(core),
        alt=" ".join(alt_core),
        legal=" ".join(sorted(set(legal))),
        skel=" ".join(skeleton(t) for t in core),
        acronym=acronym,
    )


# --------------------------------------------------------------------------------------------
# Addresses
# --------------------------------------------------------------------------------------------


@dataclass(frozen=True)
class NormAddress:
    norm: str  # full canonical address
    core: str  # landmark phrase + postal code removed
    landmark: str  # canonical landmark phrase ("near sbi atm"), "" if none
    postal: str  # 5-6 digit postal/ZIP/PIN code, "" if none
    nums: str  # space-joined sorted unique digit runs (postal excluded)
    house: str  # first number of the core address, "" if none


def _canon_address_tokens(text: str) -> list[str]:
    text = _clean(text)
    for rx, rep in _ADDR_PHRASE_RES:
        text = rx.sub(rep, text)
    toks = _join_initials(text.split())
    toks = [
        t for i, t in enumerate(toks) if not (t in _NUMBER_WORDS and i + 1 < len(toks) and toks[i + 1][:1].isdigit())
    ]
    return [t for t in _map_tokens(toks, _ADDR_MAP) if t not in _ADDR_DROP]


def _find_postal(tokens: list[str]) -> str:
    """Last 5-6 digit token. A leading one is usually a (US) house number, so it only counts as
    postal when it is 6 digits or directly followed by another number (reordered components,
    e.g. ``33005 50 impasse voltaire``)."""
    for i in range(len(tokens) - 1, 0, -1):
        if _POSTAL_RE.match(tokens[i]):
            return tokens[i]
    if tokens and _POSTAL_RE.match(tokens[0]):
        if len(tokens[0]) == 6 or (len(tokens) > 1 and tokens[1].isdigit()):
            return tokens[0]
    return ""


@cache
def normalize_address(raw: str) -> NormAddress:
    folded = fold_unicode(raw)
    folded = _PIN_SPLIT_RE.sub(r"\1\2", folded.strip())
    landmark_raw = " ".join(m.group(0) for m in _LANDMARK_RE.finditer(folded))
    without_landmark = _LANDMARK_RE.sub(" ", folded)

    full = _canon_address_tokens(folded)
    core = _canon_address_tokens(without_landmark)
    landmark = _canon_address_tokens(landmark_raw) if landmark_raw else []

    postal = _find_postal(core)
    if postal:
        core = [t for t in core if t != postal]
    digits = [t for t in core if t.isdigit()]
    nums = sorted(set(digits), key=lambda x: (len(x), x))
    return NormAddress(
        norm=" ".join(full),
        core=" ".join(core),
        landmark=" ".join(landmark),
        postal=postal,
        nums=" ".join(nums),
        house=digits[0] if digits else "",
    )


# --------------------------------------------------------------------------------------------
# DataFrame API
# --------------------------------------------------------------------------------------------

NORM_COLUMNS = (
    "name_norm",
    "name_core",
    "name_alt",
    "name_legal",
    "name_skel",
    "name_acronym",
    "addr_norm",
    "addr_core",
    "addr_landmark",
    "postal",
    "nums",
    "house",
    "country_norm",
)


def normalize_records(df: pd.DataFrame) -> pd.DataFrame:
    """Return ``df`` with the normalised columns in :data:`NORM_COLUMNS` appended."""
    n = len(df)
    names = [normalize_name(x) for x in progress(df["business_name"].tolist(), "normalise names", n, "rec")]
    addrs = [normalize_address(x) for x in progress(df["business_address"].tolist(), "normalise addresses", n, "rec")]
    out = df.copy()
    out["name_norm"] = [n.norm for n in names]
    out["name_core"] = [n.core for n in names]
    out["name_alt"] = [n.alt for n in names]
    out["name_legal"] = [n.legal for n in names]
    out["name_skel"] = [n.skel for n in names]
    out["name_acronym"] = [n.acronym for n in names]
    out["addr_norm"] = [a.norm for a in addrs]
    out["addr_core"] = [a.core for a in addrs]
    out["addr_landmark"] = [a.landmark for a in addrs]
    out["postal"] = [a.postal for a in addrs]
    out["nums"] = [a.nums for a in addrs]
    out["house"] = [a.house for a in addrs]
    out["country_norm"] = [normalize_country(x) for x in df["country"].tolist()]
    return out

In [ ]:
%%writefile er_src/er_common/similarity.py
"""Vectorised pairwise similarity primitives over aligned (left[i], right[i]) string pairs.

Everything returns float32 arrays in [0, 1] (NaN where a field is missing on either side, so
downstream models can distinguish "different" from "unknown").
"""

from __future__ import annotations

import os
from collections.abc import Callable, Sequence
from typing import Any

import numpy as np
import numpy.typing as npt
import scipy.sparse as sp
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler, Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer

from er_common.progress import progress

F32 = npt.NDArray[np.float32]
IntArr = npt.NDArray[np.int64]
Texts = Sequence[str] | npt.NDArray[Any]  # lists or pandas-derived object arrays of str

WORKERS = int(os.environ.get("ER_WORKERS", "-1"))

SCORERS: dict[str, tuple[Callable[..., float], float]] = {
    "ratio": (fuzz.ratio, 100.0),
    "partial": (fuzz.partial_ratio, 100.0),
    "tsort": (fuzz.token_sort_ratio, 100.0),
    "tset": (fuzz.token_set_ratio, 100.0),
    "jw": (JaroWinkler.normalized_similarity, 1.0),
    "lev": (Levenshtein.normalized_similarity, 1.0),
}


PAIR_CHUNK = 500_000  # pairs per rapidfuzz call; also the progress-bar granularity


def paired(left: Texts, right: Texts, scorer: str, missing_nan: bool = True) -> F32:
    fn, scale = SCORERS[scorer]
    n = len(left)
    out = np.empty(n, dtype=np.float32)
    starts = range(0, n, PAIR_CHUNK)
    it = progress(starts, f"fuzzy {scorer}", len(starts), "chunk") if n > PAIR_CHUNK else starts
    for s in it:
        e = min(s + PAIR_CHUNK, n)
        out[s:e] = process.cpdist(list(left[s:e]), list(right[s:e]), scorer=fn, workers=WORKERS, dtype=np.float32)
    out /= np.float32(scale)
    if missing_nan:
        empty = np.fromiter((not a or not b for a, b in zip(left, right, strict=True)), bool, len(left))
        out[empty] = np.nan
    return out


def rowwise_dot(a: sp.csr_matrix, b: sp.csr_matrix, ia: IntArr, ib: IntArr, chunk: int = 200_000) -> F32:
    """sum_k a[ia[n], k] * b[ib[n], k] for every n, chunked to bound memory."""
    out = np.empty(len(ia), dtype=np.float32)
    starts = range(0, len(ia), chunk)
    for s in progress(starts, "row dot products", len(starts), "chunk") if len(ia) > chunk else starts:
        e = min(s + chunk, len(ia))
        prod = a[ia[s:e]].multiply(b[ib[s:e]])
        out[s:e] = np.asarray(prod.sum(axis=1)).ravel()
    return out


class TfidfSpace:
    """TF-IDF space fitted on the union of both sides (label-free, so fitting on test text is fine)."""

    def __init__(self, analyzer: str, ngram_range: tuple[int, int], min_df: int = 1):
        extra = {"token_pattern": r"(?u)\b\w+\b"} if analyzer == "word" else {}
        self.vec = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=True,
            dtype=np.float32,
            lowercase=False,
            **extra,
        )

    def fit_transform(self, left: Texts, right: Texts) -> tuple[sp.csr_matrix, sp.csr_matrix]:
        self.vec.fit(list(left) + list(right))
        return self.vec.transform(list(left)).tocsr(), self.vec.transform(list(right)).tocsr()


def cosine_pairs(a: sp.csr_matrix, b: sp.csr_matrix, ia: IntArr, ib: IntArr) -> F32:
    """Cosine for L2-normalised TF-IDF rows; NaN where either row is empty."""
    out = rowwise_dot(a, b, ia, ib)
    empty = (np.diff(a.indptr)[ia] == 0) | (np.diff(b.indptr)[ib] == 0)
    out[empty] = np.nan
    return out


class TokenSets:
    """Binary token incidence for two sides + IDF weights -> set similarities as sparse algebra."""

    def __init__(self, left: Texts, right: Texts):
        vocab: dict[str, int] = {}
        rows_l = [self._ids(t, vocab) for t in left]
        rows_r = [self._ids(t, vocab) for t in right]
        self.left = self._matrix(rows_l, len(vocab))
        self.right = self._matrix(rows_r, len(vocab))
        df = np.asarray((self.left > 0).sum(axis=0)).ravel() + np.asarray((self.right > 0).sum(axis=0)).ravel()
        n_docs = len(rows_l) + len(rows_r)
        self.idf = np.log((n_docs + 1) / (df + 1)).astype(np.float32) + np.float32(1.0)
        self.df = df
        self.left_w = (self.left @ sp.diags(self.idf)).tocsr()
        self.right_w = (self.right @ sp.diags(self.idf)).tocsr()
        self.len_l = np.asarray(self.left.sum(axis=1)).ravel().astype(np.float32)
        self.len_r = np.asarray(self.right.sum(axis=1)).ravel().astype(np.float32)
        self.wlen_l = np.asarray(self.left_w.sum(axis=1)).ravel().astype(np.float32)
        self.wlen_r = np.asarray(self.right_w.sum(axis=1)).ravel().astype(np.float32)

    @staticmethod
    def _ids(text: str, vocab: dict[str, int]) -> list[int]:
        return sorted({vocab.setdefault(t, len(vocab)) for t in text.split()})

    @staticmethod
    def _matrix(rows: list[list[int]], n_cols: int) -> sp.csr_matrix:
        indptr = np.r_[0, np.cumsum([len(r) for r in rows])]
        indices = np.fromiter((i for r in rows for i in r), dtype=np.int64, count=int(indptr[-1]))
        data = np.ones(len(indices), dtype=np.float32)
        return sp.csr_matrix((data, indices, indptr), shape=(len(rows), max(n_cols, 1)))

    def features(self, ia: IntArr, ib: IntArr) -> dict[str, F32]:
        inter = rowwise_dot(self.left, self.right, ia, ib)
        winter = rowwise_dot(self.left_w, self.right, ia, ib)
        la, lb = self.len_l[ia], self.len_r[ib]
        wa, wb = self.wlen_l[ia], self.wlen_r[ib]
        union = la + lb - inter
        wunion = wa + wb - winter
        missing = (la == 0) | (lb == 0)
        with np.errstate(divide="ignore", invalid="ignore"):
            out = {
                "jaccard": np.where(missing, np.nan, inter / np.maximum(union, 1)),
                "wjaccard": np.where(missing, np.nan, winter / np.maximum(wunion, 1e-6)),
                "containment": np.where(missing, np.nan, inter / np.maximum(np.minimum(la, lb), 1)),
                "common": inter,
                "only_left": la - inter,
                "only_right": lb - inter,
            }
        return {k: np.asarray(v, dtype=np.float32) for k, v in out.items()}


def sparse_topk(queries: sp.csr_matrix, docs: sp.csr_matrix, k: int, max_cells: int = 20_000_000) -> tuple[IntArr, F32]:
    """Exact top-k by dot product for every query row; returns (indices, scores) of shape (n_q, k').

    Chunked dense scoring keeps memory at ``max_cells`` float32 cells regardless of corpus size.
    Rows are ordered by descending score, ties by ascending doc index (deterministic).
    """
    n_q, n_d = queries.shape[0], docs.shape[0]
    k = min(k, n_d)
    idx_out = np.zeros((n_q, k), dtype=np.int64)
    sc_out = np.zeros((n_q, k), dtype=np.float32)
    if k == 0 or n_q == 0:
        return idx_out, sc_out
    docs_t = docs.T.tocsc()
    chunk = max(1, min(n_q, max_cells // max(n_d, 1)))
    for s in progress(range(0, n_q, chunk), f"top-{k} retrieval", (n_q + chunk - 1) // chunk, "chunk"):
        e = min(s + chunk, n_q)
        dense = (queries[s:e] @ docs_t).toarray().astype(np.float32, copy=False)
        part = np.argpartition(-dense, kth=k - 1, axis=1)[:, :k] if k < n_d else np.tile(np.arange(n_d), (e - s, 1))
        part_scores = np.take_along_axis(dense, part, axis=1)
        order = np.lexsort((part, -part_scores), axis=1)
        idx_out[s:e] = np.take_along_axis(part, order, axis=1)
        sc_out[s:e] = np.take_along_axis(part_scores, order, axis=1)
    return idx_out, sc_out


def acronym_match(names_a: Texts, acr_a: Texts, names_b: Texts, acr_b: Texts) -> F32:
    """1 if one side's whole core name is a token equal to the other side's initials (SBI vs State Bank of India)."""
    out = np.zeros(len(names_a), dtype=np.float32)
    for i, (na, aa, nb, ab) in enumerate(zip(names_a, acr_a, names_b, acr_b, strict=True)):
        ta, tb = set(na.split()), set(nb.split())
        if (ab and len(ab) >= 2 and ab in ta) or (aa and len(aa) >= 2 and aa in tb):
            out[i] = 1.0
    return out


def tri_state(left: Texts, right: Texts) -> F32:
    """1 equal, 0 different, NaN if missing on either side."""
    out = np.full(len(left), np.nan, dtype=np.float32)
    for i, (a, b) in enumerate(zip(left, right, strict=True)):
        if a and b:
            out[i] = 1.0 if a == b else 0.0
    return out


def prefix_match(left: Texts, right: Texts, n: int) -> F32:
    out = np.full(len(left), np.nan, dtype=np.float32)
    for i, (a, b) in enumerate(zip(left, right, strict=True)):
        if a and b:
            out[i] = 1.0 if a[:n] == b[:n] else 0.0
    return out

In [ ]:
%%writefile er_src/er_common/metrics.py
"""Entity-level F-beta exactly as the leaderboard computes it, plus a vectorised evaluator.

Per Source 1 entity with true set T and predicted set P (beta = 0.5):
    T = {}, P = {}   -> 1.0
    T = {}, P != {}  -> 0.0
    T != {}, P = {}  -> 0.0
    otherwise        -> (1 + b^2) |P n T| / (|P| + b^2 |T|)
and the score is the unweighted mean over entities.

Note that ``|T|`` always counts *all* ground-truth matches, including those the blocking stage
never generated -- recall lost in candidate generation is charged to the final score.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from dataclasses import asdict, dataclass

import numpy as np
import numpy.typing as npt

from er_common.progress import progress

BETA = 0.5

FloatArr = npt.NDArray[np.float64]
IntArr = npt.NDArray[np.int64]
BoolArr = npt.NDArray[np.bool_]


def f_beta_counts(n_tp: int, n_pred: int, n_true: int, beta: float = BETA) -> float:
    if n_true == 0:
        return 1.0 if n_pred == 0 else 0.0
    if n_pred == 0 or n_tp == 0:
        return 0.0
    b2 = beta * beta
    return (1.0 + b2) * n_tp / (n_pred + b2 * n_true)


@dataclass(frozen=True)
class EvalReport:
    f05: float
    macro_precision: float  # over entities with >= 1 prediction
    macro_recall: float  # over entities with >= 1 true match
    pair_precision: float
    pair_recall: float
    singleton_accuracy: float  # share of true singletons predicted empty
    matched_entity_f05: float  # F0.5 restricted to entities with >= 1 true match
    false_merge_entities: int  # entities with >= 1 wrong id predicted
    n_entities: int
    n_singletons: int
    n_pred_pairs: int
    n_true_pairs: int

    def as_dict(self) -> dict[str, float | int]:
        return asdict(self)

    def summary(self) -> str:
        return (
            f"F0.5={self.f05:.4f}  P(macro)={self.macro_precision:.4f}  R(macro)={self.macro_recall:.4f}  "
            f"pairP={self.pair_precision:.4f}  pairR={self.pair_recall:.4f}  "
            f"singletonAcc={self.singleton_accuracy:.4f}  matchedF0.5={self.matched_entity_f05:.4f}  "
            f"falseMergeEntities={self.false_merge_entities}  n={self.n_entities}"
        )


def evaluate(
    pred: Mapping[str, Sequence[str]],
    gt: Mapping[str, Sequence[str]],
    s1_ids: Sequence[str],
    beta: float = BETA,
) -> EvalReport:
    f_all: list[float] = []
    f_matched: list[float] = []
    precs: list[float] = []
    recs: list[float] = []
    tp_tot = pred_tot = true_tot = 0
    n_single = single_ok = false_merge = 0
    for s1 in progress(s1_ids, "score entities", len(s1_ids), "S1"):
        p = set(pred.get(s1, ()))
        t = set(gt.get(s1, ()))
        tp = len(p & t)
        f = f_beta_counts(tp, len(p), len(t), beta)
        f_all.append(f)
        tp_tot += tp
        pred_tot += len(p)
        true_tot += len(t)
        if p:
            precs.append(tp / len(p))
            false_merge += int(tp < len(p))
        if t:
            recs.append(tp / len(t))
            f_matched.append(f)
        else:
            n_single += 1
            single_ok += int(not p)
    return EvalReport(
        f05=float(np.mean(f_all)) if f_all else 0.0,
        macro_precision=float(np.mean(precs)) if precs else 1.0,
        macro_recall=float(np.mean(recs)) if recs else 1.0,
        pair_precision=tp_tot / pred_tot if pred_tot else 1.0,
        pair_recall=tp_tot / true_tot if true_tot else 1.0,
        singleton_accuracy=single_ok / n_single if n_single else 1.0,
        matched_entity_f05=float(np.mean(f_matched)) if f_matched else 1.0,
        false_merge_entities=false_merge,
        n_entities=len(s1_ids),
        n_singletons=n_single,
        n_pred_pairs=pred_tot,
        n_true_pairs=true_tot,
    )


def candidate_recall(
    candidates: Mapping[str, Sequence[str]], gt: Mapping[str, Sequence[str]], s1_ids: Sequence[str]
) -> dict[str, float]:
    """Blocking diagnostics: pair recall, share of entities whose full true set was retrieved,
    and the F0.5 ceiling an oracle matcher could reach with these candidates."""
    hit = total = full = 0
    ceiling: list[float] = []
    sizes: list[int] = []
    for s1 in progress(s1_ids, "score entities", len(s1_ids), "S1"):
        c = set(candidates.get(s1, ()))
        t = set(gt.get(s1, ()))
        sizes.append(len(c))
        found = len(c & t)
        hit += found
        total += len(t)
        full += int(found == len(t))
        ceiling.append(f_beta_counts(found, found, len(t)))
    n = max(len(s1_ids), 1)
    return {
        "pair_recall": hit / total if total else 1.0,
        "entity_full_recall": full / n,
        "oracle_f05_ceiling": float(np.mean(ceiling)) if ceiling else 1.0,
        "mean_candidates": float(np.mean(sizes)) if sizes else 0.0,
        "max_candidates": float(max(sizes, default=0)),
    }


class GroupedEvaluator:
    """Fast macro F-beta for boolean masks over a fixed pair table (used inside tuning loops).

    ``group``   : int group id per pair (S1 index 0..G-1)
    ``label``   : bool per pair
    ``n_true``  : int per group -- full ground-truth count (includes matches missed by blocking)
    ``active``  : bool per group -- which S1 entities count towards the score
    """

    def __init__(self, group: IntArr, label: BoolArr, n_true: IntArr, active: BoolArr, beta: float = BETA):
        self.group = np.asarray(group, dtype=np.int64)
        self.label = np.asarray(label, dtype=bool)
        self.n_true = np.asarray(n_true, dtype=np.float64)
        self.active = np.asarray(active, dtype=bool)
        self.n_groups = len(self.n_true)
        self.b2 = beta * beta
        if self.active.sum() == 0:
            raise ValueError("GroupedEvaluator needs at least one active group")

    def per_group(self, mask: BoolArr) -> FloatArr:
        g = self.group[mask]
        k = np.bincount(g, minlength=self.n_groups).astype(np.float64)
        tp = np.bincount(g, weights=self.label[mask].astype(np.float64), minlength=self.n_groups)
        denom = k + self.b2 * self.n_true
        f = np.where(tp > 0, (1.0 + self.b2) * tp / np.where(denom > 0, denom, 1.0), 0.0)
        f = np.where(self.n_true == 0, (k == 0).astype(np.float64), f)
        return f

    def score(self, mask: BoolArr) -> float:
        return float(self.per_group(mask)[self.active].mean())

In [ ]:
%%writefile er_src/er_common/decision.py
"""Decision layer: turn pair scores into per-entity match sets.

Three building blocks, all vectorised over a flat pair table (``group`` = S1 index):

1. ``exclusive_mask``      -- an S2/S3 record can belong to at most one S1 entity (S1 is
                              deduplicated). Keep only the best-scoring S1 per candidate.
2. ``threshold_margin``    -- absolute threshold ``tau`` + relative margin ``delta`` to the
                              entity's best candidate + optional ``max_k``.
3. ``expected_fbeta_topk`` -- Bayes-optimal set under calibrated, independent probabilities:
                              for each entity pick k maximising E[F_beta(top-k)]. Singletons
                              fall out naturally (k = 0 wins when P(no match) dominates).
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import numpy.typing as npt

from er_common.metrics import BETA

FloatArr = npt.NDArray[np.float64]
IntArr = npt.NDArray[np.int64]
BoolArr = npt.NDArray[np.bool_]


def group_max(group: IntArr, score: FloatArr, n_groups: int) -> FloatArr:
    out = np.full(n_groups, -np.inf)
    np.maximum.at(out, group, score)
    return out


def group_rank(group: IntArr, score: FloatArr) -> IntArr:
    """0-based rank of each pair inside its group by descending score (ties broken by position)."""
    order = np.lexsort((np.arange(len(score)), -score, group))
    g_sorted = group[order]
    starts = np.r_[0, np.flatnonzero(np.diff(g_sorted)) + 1]
    run_start = np.repeat(starts, np.diff(np.r_[starts, len(order)]))
    rank = np.empty(len(score), dtype=np.int64)
    rank[order] = np.arange(len(order)) - run_start
    return rank


def exclusive_mask(cand: IntArr, score: FloatArr, eligible: BoolArr | None = None) -> BoolArr:
    """True for pairs holding the highest score for their candidate record among eligible pairs."""
    n = len(score)
    elig = np.ones(n, dtype=bool) if eligible is None else eligible
    idx = np.flatnonzero(elig)
    keep = np.zeros(n, dtype=bool)
    if idx.size == 0:
        return keep
    order = idx[np.lexsort((idx, -score[idx], cand[idx]))]
    c_sorted = cand[order]
    first = np.r_[True, c_sorted[1:] != c_sorted[:-1]]
    keep[order[first]] = True
    return keep


def threshold_margin(
    group: IntArr,
    score: FloatArr,
    n_groups: int,
    tau: float,
    delta: float,
    max_k: int = 0,
    base: BoolArr | None = None,
) -> BoolArr:
    """Pairs with ``score >= tau`` and ``score >= best_in_group - delta`` (and rank < max_k if > 0)."""
    eligible = np.ones(len(score), dtype=bool) if base is None else base
    s = np.where(eligible, score, -np.inf)
    gmax = group_max(group, s, n_groups)
    mask = eligible & (s >= tau) & (s >= gmax[group] - delta)
    if max_k > 0:
        mask &= group_rank(group, s) < max_k
    return mask


def _pb_pmf(p: FloatArr) -> FloatArr:
    pmf = np.ones(1)
    for x in p:
        pmf = np.convolve(pmf, np.array([1.0 - x, x]))
    return pmf


def expected_fbeta_best_k(p_sorted: FloatArr, beta: float = BETA, k_max: int = 20) -> tuple[int, float]:
    """Best k (and its expected F) for candidate probabilities sorted in descending order."""
    b2 = beta * beta
    m = len(p_sorted)
    best_k, best_e = 0, float(np.prod(1.0 - p_sorted))
    if m == 0:
        return 0, 1.0
    suffix: list[FloatArr] = [np.ones(1)] * (m + 1)
    for i in range(m - 1, -1, -1):
        suffix[i] = np.convolve(suffix[i + 1], np.array([1.0 - p_sorted[i], p_sorted[i]]))
    pin = np.ones(1)
    for k in range(1, min(m, k_max) + 1):
        pin = np.convolve(pin, np.array([1.0 - p_sorted[k - 1], p_sorted[k - 1]]))
        pout = suffix[k]
        a = np.arange(k + 1, dtype=np.float64)[:, None]
        b = np.arange(len(pout), dtype=np.float64)[None, :]
        val = (1.0 + b2) * a / (k + b2 * (a + b))
        e = float(pin @ val @ pout)
        if e > best_e + 1e-12:
            best_k, best_e = k, e
    return best_k, best_e


def expected_fbeta_topk(
    group: IntArr,
    prob: FloatArr,
    n_groups: int,
    beta: float = BETA,
    min_prob: float = 1e-3,
    k_max: int = 20,
    base: BoolArr | None = None,
) -> BoolArr:
    """Per group, predict the top-k pairs maximising expected F-beta (labels assumed independent)."""
    eligible = np.ones(len(prob), dtype=bool) if base is None else base
    p = np.where(eligible, prob, 0.0)
    mask = np.zeros(len(p), dtype=bool)
    idx = np.flatnonzero(p >= min_prob)
    if idx.size == 0:
        return mask
    order = idx[np.lexsort((idx, -p[idx], group[idx]))]
    g_sorted = group[order]
    starts = np.r_[0, np.flatnonzero(np.diff(g_sorted)) + 1]
    ends = np.r_[starts[1:], len(order)]
    for s, e in zip(starts, ends, strict=True):
        rows = order[s:e]
        if p[rows[0]] < 0.05:  # E[F|k>=1] <= p_top < P(no match) for any realistic list
            continue
        k, _ = expected_fbeta_best_k(p[rows], beta, k_max)
        mask[rows[:k]] = True
    return mask


@dataclass(frozen=True)
class PlattCalibrator:
    """p' = sigmoid(a * logit(p) + b), fitted by logistic regression on out-of-fold scores."""

    a: float = 1.0
    b: float = 0.0

    @staticmethod
    def _logit(p: FloatArr) -> FloatArr:
        q = np.clip(p, 1e-6, 1 - 1e-6)
        return np.log(q / (1 - q))

    @classmethod
    def fit(cls, prob: FloatArr, label: BoolArr) -> PlattCalibrator:
        from sklearn.linear_model import LogisticRegression

        x = cls._logit(np.asarray(prob, dtype=np.float64)).reshape(-1, 1)
        y = np.asarray(label, dtype=int)
        if y.min() == y.max():
            return cls()
        lr = LogisticRegression(C=1e4, solver="lbfgs", max_iter=1000)
        lr.fit(x, y)
        return cls(a=float(lr.coef_[0, 0]), b=float(lr.intercept_[0]))

    def __call__(self, prob: FloatArr) -> FloatArr:
        z = self.a * self._logit(np.asarray(prob, dtype=np.float64)) + self.b
        return 1.0 / (1.0 + np.exp(-z))

In [ ]:
%%writefile er_src/er_common/folds.py
"""Deterministic S1-level fold assignment.

Folds are over Source 1 entities (the unit the metric averages over), stratified by
(country, number-of-true-matches bucket) so each fold sees the same singleton / multi-match mix.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence

import numpy as np


def stratified_s1_folds(
    s1_ids: Sequence[str],
    countries: Sequence[str],
    gt: Mapping[str, Sequence[str]],
    n_folds: int,
    seed: int,
) -> dict[str, int]:
    if n_folds < 2:
        raise ValueError("n_folds must be >= 2")
    strata: dict[tuple[str, int], list[str]] = {}
    for s1, country in zip(s1_ids, countries, strict=True):
        key = (country, min(len(gt.get(s1, ())), 2))
        strata.setdefault(key, []).append(s1)
    rng = np.random.default_rng(seed)
    folds: dict[str, int] = {}
    offset = 0
    for key in sorted(strata):
        members = sorted(strata[key])
        perm = rng.permutation(len(members))
        for i, j in enumerate(perm):
            folds[members[j]] = (i + offset) % n_folds
        offset += len(members)
    return folds

In [ ]:
%%writefile er_src/er_common/tuning.py
"""Derivative-free parameter search against the (non-differentiable) entity-level F0.5.

Random search over the box, seeded with the hand-set defaults, followed by coordinate
refinement on a shrinking grid. Fully deterministic given ``seed``.
"""

from __future__ import annotations

from collections.abc import Callable, Mapping
from dataclasses import dataclass

import numpy as np

from er_common.progress import bar

Params = dict[str, float]


@dataclass(frozen=True)
class Dim:
    low: float
    high: float
    integer: bool = False

    def clip(self, x: float) -> float:
        x = min(max(x, self.low), self.high)
        return float(round(x)) if self.integer else float(x)

    def sample(self, rng: np.random.Generator) -> float:
        return self.clip(rng.uniform(self.low, self.high))


def search(
    objective: Callable[[Params], float],
    space: Mapping[str, Dim],
    start: Params,
    n_random: int = 400,
    n_rounds: int = 3,
    grid: int = 9,
    seed: int = 0,
) -> tuple[Params, float]:
    rng = np.random.default_rng(seed)
    best = {k: space[k].clip(v) for k, v in start.items()}
    total = 1 + n_random + n_rounds * len(space) * grid
    with bar("tune parameters", total, "eval") as pb:
        best_score = objective(best)
        pb.update(1)
        for _ in range(n_random):
            cand = {k: d.sample(rng) for k, d in space.items()}
            s = objective(cand)
            if s > best_score + 1e-12:
                best, best_score = cand, s
            pb.update(1)
            pb.set_postfix_str(f"best F0.5={best_score:.4f}", refresh=False)
        width = 0.5
        for _ in range(n_rounds):
            for k in sorted(space):
                d = space[k]
                span = (d.high - d.low) * width
                for v in np.linspace(best[k] - span / 2, best[k] + span / 2, grid):
                    pb.update(1)
                    cand = dict(best)
                    cand[k] = d.clip(float(v))
                    if cand[k] == best[k]:
                        continue
                    s = objective(cand)
                    if s > best_score + 1e-12:
                        best, best_score = cand, s
                        pb.set_postfix_str(f"best F0.5={best_score:.4f}", refresh=False)
            width *= 0.5
    return best, best_score

In [ ]:
%%writefile er_src/er_common/pairs.py
"""Candidate pair table shared by all methods."""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field

import numpy as np
import numpy.typing as npt
import pandas as pd

from er_common.progress import progress

IntArr = npt.NDArray[np.int64]
BoolArr = npt.NDArray[np.bool_]
FloatArr = npt.NDArray[np.float64]


@dataclass
class PairTable:
    """Flat (S1 record, S2/S3 record) candidate list plus per-pair features.

    ``s1`` / ``s23`` are the normalised record frames; ``s1_idx`` / ``c_idx`` index their rows.
    Pairs are unique and sorted by (s1_idx, c_idx).
    """

    s1: pd.DataFrame
    s23: pd.DataFrame
    s1_idx: IntArr
    c_idx: IntArr
    feats: pd.DataFrame = field(default_factory=pd.DataFrame)

    def __post_init__(self) -> None:
        if len(self.s1_idx) != len(self.c_idx):
            raise ValueError("s1_idx and c_idx must be aligned")
        if len(self.feats) and len(self.feats) != len(self.s1_idx):
            raise ValueError("feature frame not aligned with pairs")

    @property
    def n_pairs(self) -> int:
        return len(self.s1_idx)

    @property
    def n_s1(self) -> int:
        return len(self.s1)

    @property
    def s1_ids(self) -> list[str]:
        return self.s1["entity_id"].tolist()

    @property
    def c_ids(self) -> npt.NDArray[np.str_]:
        return self.s23["entity_id"].to_numpy()

    def labels(self, gt: Mapping[str, Sequence[str]]) -> BoolArr:
        s1_ids = self.s1["entity_id"].to_numpy()
        c_ids = self.c_ids
        true_pairs = {(s, c) for s, cs in gt.items() for c in cs}
        return np.fromiter(
            (
                (s1_ids[i], c_ids[j]) in true_pairs
                for i, j in progress(zip(self.s1_idx, self.c_idx, strict=True), "label pairs", self.n_pairs, "pair")
            ),
            dtype=bool,
            count=self.n_pairs,
        )

    def n_true(self, gt: Mapping[str, Sequence[str]]) -> IntArr:
        return np.array([len(gt.get(s, ())) for s in self.s1["entity_id"]], dtype=np.int64)

    def id_lists(self, mask: BoolArr | None = None, score: FloatArr | None = None) -> dict[str, list[str]]:
        """{s1_id: [cand ids]} for pairs in ``mask`` (all pairs if None), best score first."""
        sel = np.ones(self.n_pairs, dtype=bool) if mask is None else mask
        idx = np.flatnonzero(sel)
        key = -score[idx] if score is not None else np.zeros(len(idx))
        order = idx[np.lexsort((self.c_idx[idx], key, self.s1_idx[idx]))]
        s1_ids = self.s1["entity_id"].to_numpy()
        c_ids = self.c_ids
        out: dict[str, list[str]] = {s: [] for s in s1_ids}
        for i in progress(order, "collect id lists", len(order), "pair"):
            out[s1_ids[self.s1_idx[i]]].append(c_ids[self.c_idx[i]])
        return out

    def subset(self, mask: BoolArr) -> PairTable:
        return PairTable(
            s1=self.s1,
            s23=self.s23,
            s1_idx=self.s1_idx[mask],
            c_idx=self.c_idx[mask],
            feats=self.feats.loc[mask].reset_index(drop=True) if len(self.feats) else self.feats,
        )


def unique_pairs(s1_idx: IntArr, c_idx: IntArr, n_c: int) -> tuple[IntArr, IntArr]:
    code = np.unique(s1_idx.astype(np.int64) * max(n_c, 1) + c_idx.astype(np.int64))
    return code // max(n_c, 1), code % max(n_c, 1)

In [ ]:
%%writefile er_src/er_common/runner.py
"""Shared command-line driver: cross-validated evaluation on train + inference on test.

    python -m <method_package> validate --data-dir dataset --out-dir outputs/<method>
    python -m <method_package> predict  --data-dir dataset --out-dir outputs/<method>
    python -m <method_package> all      --data-dir dataset --out-dir outputs/<method>

``validate`` scores every train S1 entity with a model that never saw its labels (K-fold over
S1 entities, all tuning nested inside the fold), then writes metrics and an error dump.
``predict`` fits on all train labels and writes the two submission files for the test split.
"""

from __future__ import annotations

import argparse
import dataclasses
import hashlib
import json
import logging
import platform
import subprocess
import sys
import time
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any, Protocol

import numpy as np
import numpy.typing as npt
import pandas as pd
from tqdm.contrib.logging import logging_redirect_tqdm

from er_common.folds import stratified_s1_folds
from er_common.io import (
    SplitData,
    candidate_owner_multiplicity,
    gt_for,
    load_ground_truth,
    load_split,
    write_id_lists,
    write_submission,
)
from er_common.metrics import candidate_recall, evaluate
from er_common.pairs import PairTable
from er_common.progress import progress, set_enabled

BoolArr = npt.NDArray[np.bool_]
FloatArr = npt.NDArray[np.float64]
IntArr = npt.NDArray[np.int64]
GT = Mapping[str, Sequence[str]]

log = logging.getLogger("er")


class Method(Protocol):
    name: str
    config: Any

    def build_pairs(self, data: SplitData) -> PairTable: ...

    def fit(self, pt: PairTable, gt: GT, active_s1: BoolArr) -> Any: ...

    def predict(self, model: Any, pt: PairTable) -> tuple[BoolArr, FloatArr]: ...

    def model_summary(self, model: Any) -> dict[str, Any]: ...


def generic_cross_validate(
    method: Method, pt: PairTable, gt: GT, folds: IntArr, n_folds: int
) -> tuple[BoolArr, FloatArr, list[dict[str, Any]]]:
    pred = np.zeros(pt.n_pairs, dtype=bool)
    score = np.zeros(pt.n_pairs, dtype=np.float64)
    summaries: list[dict[str, Any]] = []
    for k in progress(range(n_folds), "cv folds", n_folds, "fold"):
        t0 = time.perf_counter()
        model = method.fit(pt, gt, folds != k)
        mask, sc = method.predict(model, pt)
        in_fold = folds[pt.s1_idx] == k
        pred[in_fold] = mask[in_fold]
        score[in_fold] = sc[in_fold]
        summaries.append({"fold": k, "seconds": round(time.perf_counter() - t0, 2), **method.model_summary(model)})
        log.info("fold %d/%d done in %.1fs", k + 1, n_folds, time.perf_counter() - t0)
    return pred, score, summaries


# ------------------------------------------------------------------------------------------------
# helpers
# ------------------------------------------------------------------------------------------------


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _git_commit() -> str:
    try:
        return subprocess.run(
            ["git", "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
            cwd=Path(__file__).resolve().parents[1],
        ).stdout.strip()
    except Exception:
        return "unknown"


def _versions() -> dict[str, str]:
    import importlib.metadata as md

    out = {"python": platform.python_version()}
    for pkg in ("numpy", "pandas", "scipy", "scikit-learn", "rapidfuzz", "lightgbm"):
        try:
            out[pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            out[pkg] = "not installed"
    return out


def _manifest(method: Method, data_dir: Path, split: str, extra: dict[str, Any]) -> dict[str, Any]:
    files = sorted((data_dir / split).glob(f"{split}_*.tsv"))
    return {
        "method": method.name,
        "config": (
            dataclasses.asdict(method.config)
            if dataclasses.is_dataclass(method.config) and not isinstance(method.config, type)
            else {}
        ),
        "git_commit": _git_commit(),
        "versions": _versions(),
        "platform": platform.platform(),
        "data": {f.name: _sha256(f) for f in files},
        **extra,
    }


def _json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=False, default=str) + "\n", encoding="utf-8")


def _error_dump(pt: PairTable, gt: GT, pred: BoolArr, score: FloatArr, path: Path) -> None:
    label = pt.labels(gt)
    s1_ids = pt.s1["entity_id"].to_numpy()
    rows = []
    for kind, sel in (("false_positive", pred & ~label), ("false_negative", ~pred & label)):
        idx = np.flatnonzero(sel)
        for i in progress(idx, f"error dump ({kind})", len(idx), "pair"):
            a, b = pt.s1.iloc[pt.s1_idx[i]], pt.s23.iloc[pt.c_idx[i]]
            rows.append(
                {
                    "kind": kind,
                    "score": round(float(score[i]), 4),
                    "s1_id": a["entity_id"],
                    "cand_id": b["entity_id"],
                    "s1_name": a["business_name"],
                    "cand_name": b["business_name"],
                    "s1_address": a["business_address"],
                    "cand_address": b["business_address"],
                    "s1_country": a["country"],
                    "cand_country": b["country"],
                }
            )
    retrieved = {(s1_ids[i], pt.c_ids[j]) for i, j in zip(pt.s1_idx, pt.c_idx, strict=True)}
    c_lookup = pt.s23.set_index("entity_id")
    s1_lookup = pt.s1.set_index("entity_id")
    for s1, cs in progress(gt.items(), "error dump (blocking misses)", len(gt), "S1"):
        for c in cs:
            if (s1, c) not in retrieved:
                a, b = s1_lookup.loc[s1], c_lookup.loc[c]
                rows.append(
                    {
                        "kind": "missed_by_blocking",
                        "score": float("nan"),
                        "s1_id": s1,
                        "cand_id": c,
                        "s1_name": a["business_name"],
                        "cand_name": b["business_name"],
                        "s1_address": a["business_address"],
                        "cand_address": b["business_address"],
                        "s1_country": a["country"],
                        "cand_country": b["country"],
                    }
                )
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["kind", "score"], ascending=[True, False])
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)


def _per_country(pt: PairTable, pred: Mapping[str, Sequence[str]], gt: GT) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    for country, grp in pt.s1.groupby("country_norm", sort=True):
        ids = grp["entity_id"].tolist()
        rep = evaluate(pred, gt, ids)
        out[str(country) or "<empty>"] = {
            "n": len(ids),
            "f05": round(rep.f05, 4),
            "singleton_acc": round(rep.singleton_accuracy, 4),
            "pair_precision": round(rep.pair_precision, 4),
            "pair_recall": round(rep.pair_recall, 4),
        }
    return out


# ------------------------------------------------------------------------------------------------
# commands
# ------------------------------------------------------------------------------------------------


def load_train(method: Method, data_dir: Path) -> tuple[PairTable, dict[str, tuple[str, ...]]]:
    t0 = time.perf_counter()
    train = load_split(data_dir, "train")
    if train.gt is None:
        raise FileNotFoundError(f"{data_dir}/train/train_ground_truth.tsv is required")
    gt = gt_for(train.gt, train.s1_ids)
    log.info(
        "train: %d S1, %d S2+S3 records; max S1 owners per S2/S3 record in GT = %d",
        len(train.s1),
        len(train.s23),
        candidate_owner_multiplicity(gt),
    )
    pt = method.build_pairs(train)
    log.info("train pairs built in %.1fs", time.perf_counter() - t0)
    return pt, gt


def run_validate(
    method: Method,
    data_dir: Path,
    out_dir: Path,
    n_folds: int,
    seed: int,
    train: tuple[PairTable, dict[str, tuple[str, ...]]] | None = None,
) -> dict[str, Any]:
    t0 = time.perf_counter()
    pt, gt = train or load_train(method, data_dir)
    t_pairs = time.perf_counter() - t0
    blocking = candidate_recall(pt.id_lists(), gt, pt.s1_ids)
    log.info(
        "candidates: %d pairs (%.1f / S1)  pair recall %.4f  oracle F0.5 ceiling %.4f  [%.1fs]",
        pt.n_pairs,
        pt.n_pairs / max(pt.n_s1, 1),
        blocking["pair_recall"],
        blocking["oracle_f05_ceiling"],
        t_pairs,
    )

    folds_map = stratified_s1_folds(pt.s1_ids, pt.s1["country_norm"].tolist(), gt, n_folds, seed)
    folds = np.array([folds_map[s] for s in pt.s1_ids], dtype=np.int64)
    cv = getattr(method, "cross_validate", None)
    if callable(cv):
        pred_mask, score, fold_info = cv(pt, gt, folds, n_folds)
    else:
        pred_mask, score, fold_info = generic_cross_validate(method, pt, gt, folds, n_folds)

    pred = pt.id_lists(pred_mask, score)
    report = evaluate(pred, gt, pt.s1_ids)
    per_fold = []
    for k in range(n_folds):
        ids = [s for s in pt.s1_ids if folds_map[s] == k]
        per_fold.append(round(evaluate(pred, gt, ids).f05, 4))
    log.info("CV %s", report.summary())
    log.info("per-fold F0.5: %s  (std %.4f)", per_fold, float(np.std(per_fold)))

    vdir = out_dir / "validation"
    write_id_lists(vdir / "oof_matching_results.tsv", ("source1_entity_id", "matched_entity_ids"), pred, pt.s1_ids)
    write_id_lists(
        vdir / "candidate_pairs.tsv", ("source1_entity_id", "candidate_entity_ids"), pt.id_lists(), pt.s1_ids
    )
    _error_dump(pt, gt, pred_mask, score, vdir / "errors.tsv")
    result = {
        "cv": report.as_dict(),
        "per_fold_f05": per_fold,
        "per_country": _per_country(pt, pred, gt),
        "blocking": blocking,
        "n_pairs": pt.n_pairs,
        "folds": fold_info,
        "seconds": round(time.perf_counter() - t0, 1),
    }
    _json_dump(_manifest(method, data_dir, "train", {"validation": result}), vdir / "report.json")
    return result


def run_predict(
    method: Method,
    data_dir: Path,
    out_dir: Path,
    test_gt: Path | None,
    train: tuple[PairTable, dict[str, tuple[str, ...]]] | None = None,
) -> dict[str, Any]:
    t0 = time.perf_counter()
    pt_train, gt = train or load_train(method, data_dir)
    model = method.fit(pt_train, gt, np.ones(pt_train.n_s1, dtype=bool))
    log.info("fitted on all train labels: %s", json.dumps(method.model_summary(model), default=str)[:400])

    test = load_split(data_dir, "test")
    pt = method.build_pairs(test)
    mask, score = method.predict(model, pt)
    matches = pt.id_lists(mask, score)
    candidates = pt.id_lists(None, score)
    tdir = out_dir / "test"
    write_submission(tdir, matches, candidates, pt.s1_ids, pt.s23["entity_id"].tolist())
    n_nonempty = sum(1 for v in matches.values() if v)
    log.info(
        "test: %d S1, %d candidate pairs, %d predicted pairs, %d S1 with >=1 match -> %s",
        pt.n_s1,
        pt.n_pairs,
        int(mask.sum()),
        n_nonempty,
        tdir,
    )
    extra: dict[str, Any] = {
        "test": {
            "n_s1": pt.n_s1,
            "n_candidate_pairs": pt.n_pairs,
            "n_pred_pairs": int(mask.sum()),
            "n_s1_with_match": n_nonempty,
            "countries": pt.s1["country_norm"].value_counts().to_dict(),
        },
        "model": method.model_summary(model),
        "seconds": round(time.perf_counter() - t0, 1),
    }
    if test_gt is not None and test_gt.is_file():
        tgt = gt_for(load_ground_truth(test_gt), pt.s1_ids)
        rep = evaluate(matches, tgt, pt.s1_ids)
        extra["test_scored_against"] = str(test_gt)
        extra["test_eval"] = rep.as_dict()
        extra["test_per_country"] = _per_country(pt, matches, tgt)
        extra["test_blocking"] = candidate_recall(candidates, tgt, pt.s1_ids)
        log.info("TEST %s", rep.summary())
        log.info("TEST per country: %s", extra["test_per_country"])
    _json_dump(_manifest(method, data_dir, "test", extra), tdir / "manifest.json")
    return extra


def main(method: Method, argv: Sequence[str] | None = None) -> None:
    ap = argparse.ArgumentParser(
        prog=f"python -m {method.name}", description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    ap.add_argument("command", choices=("validate", "predict", "all"))
    ap.add_argument("--data-dir", type=Path, default=Path("dataset"))
    ap.add_argument("--out-dir", type=Path, default=None)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--seed", type=int, default=None, help="overrides config.seed")
    ap.add_argument("--config", type=Path, default=None, help="JSON file with config overrides")
    ap.add_argument(
        "--test-gt", type=Path, default=None, help="optional ground truth for the test split (synthetic data only)"
    )
    ap.add_argument("--no-progress", action="store_true", help="disable progress bars (e.g. when logging to a file)")
    args = ap.parse_args(argv)

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", stream=sys.stderr)
    set_enabled(not args.no_progress)
    if args.config is not None:
        overrides = json.loads(args.config.read_text(encoding="utf-8"))
        method.config = dataclasses.replace(method.config, **overrides)
    if args.seed is not None:
        method.config = dataclasses.replace(method.config, seed=args.seed)
    out_dir = args.out_dir or Path("outputs") / method.name
    log.info("%s config: %s", method.name, method.config)

    with logging_redirect_tqdm():  # log lines print above the bars instead of breaking them
        train = load_train(method, args.data_dir)
        if args.command in ("validate", "all"):
            run_validate(method, args.data_dir, out_dir, args.folds, method.config.seed, train)
        if args.command in ("predict", "all"):
            run_predict(method, args.data_dir, out_dir, args.test_gt, train)

In [ ]:
%%writefile er_src/method2_tfidf_retrieval/__init__.py
"""Method 2: TF-IDF / BM25 multi-channel retrieval + weighted similarity scoring."""

In [ ]:
%%writefile er_src/method2_tfidf_retrieval/retrieval.py
"""Method 2 candidate generation: multi-channel retrieval + reciprocal-rank fusion.

Channels (each run separately against S2 and against S3, so a source with many near-duplicate
records cannot crowd the other one out of the top-k):

    name_char  char_wb 2-4-gram TF-IDF cosine on the core name (+ trade name)
    addr_char  char_wb 3-4-gram TF-IDF cosine on the core address
    bm25       Okapi BM25 over word tokens of name + address + postal code
    postal     exact postal-code block, ranked by name_char cosine inside the block

Fusion: RRF(pair) = sum_channels 1 / (rrf_k + rank); the ``k_final`` best pairs per
(S1, source) survive. The per-channel ranks / scores are kept as features for later methods.
"""

from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field

import numpy as np
import numpy.typing as npt
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer

from er_common.progress import progress, stages
from er_common.similarity import TfidfSpace, sparse_topk

IntArr = npt.NDArray[np.int64]
F32 = npt.NDArray[np.float32]

CHANNELS = ("name_char", "addr_char", "bm25", "postal")


@dataclass(frozen=True)
class RetrievalConfig:
    k_name: int = 15
    k_addr: int = 15
    k_bm25: int = 15
    k_postal: int = 10
    k_final: int = 25
    rrf_k: float = 20.0
    bm25_k1: float = 1.2
    bm25_b: float = 0.75


def name_text(df: pd.DataFrame) -> list[str]:
    core = df["name_core"].to_numpy()
    alt = df["name_alt"].to_numpy()
    return [f"{c} {a}".strip() for c, a in zip(core, alt, strict=True)]


def bm25_text(df: pd.DataFrame) -> list[str]:
    return [f"{n} {a} {p}".strip() for n, a, p in zip(df["name_core"], df["addr_core"], df["postal"], strict=True)]


class BM25:
    def __init__(self, k1: float, b: float):
        self.k1, self.b = k1, b
        self.vec = CountVectorizer(token_pattern=r"\S+", lowercase=False, dtype=np.float32)

    def fit_transform(self, queries: list[str], docs: list[str]) -> tuple[sp.csr_matrix, sp.csr_matrix]:
        self.vec.fit(queries + docs)
        tf = self.vec.transform(docs).tocsr().astype(np.float32)
        n_docs = tf.shape[0]
        df = np.bincount(tf.indices, minlength=tf.shape[1]).astype(np.float32)
        idf = np.log1p((n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = float(dl.mean()) if n_docs else 1.0
        norm = self.k1 * (1 - self.b + self.b * dl / max(avgdl, 1e-6))
        w = tf.copy()
        rows = np.repeat(np.arange(n_docs), np.diff(w.indptr))
        w.data = idf[w.indices] * w.data * (self.k1 + 1) / (w.data + norm[rows])
        q = self.vec.transform(queries).tocsr().astype(np.float32)
        q.data[:] = 1.0
        return q, w.tocsr()


@dataclass
class RetrievalResult:
    s1_idx: IntArr
    c_idx: IntArr
    feats: pd.DataFrame
    spaces: dict[str, tuple[sp.csr_matrix, sp.csr_matrix]] = field(default_factory=dict)


def _postal_channel(
    s1: pd.DataFrame, s23: pd.DataFrame, src_rows: IntArr, q: sp.csr_matrix, d: sp.csr_matrix, k: int
) -> tuple[list[int], list[int], list[float]]:
    blocks: dict[str, list[int]] = defaultdict(list)
    postal23 = s23["postal"].to_numpy()
    for j in src_rows:
        if postal23[j]:
            blocks[postal23[j]].append(int(j))
    out_i: list[int] = []
    out_j: list[int] = []
    out_r: list[float] = []
    codes = s1["postal"].to_numpy()
    for i, code in enumerate(progress(codes, "postal block", len(codes), "rec")):
        members = blocks.get(code) if code else None
        if not members:
            continue
        m = np.asarray(members, dtype=np.int64)
        sims = np.asarray((d[m] @ q[i].T).todense()).ravel()
        order = np.lexsort((m, -sims))[:k]
        out_i += [i] * len(order)
        out_j += m[order].tolist()
        out_r += list(range(1, len(order) + 1))
    return out_i, out_j, out_r


def retrieve(s1: pd.DataFrame, s23: pd.DataFrame, cfg: RetrievalConfig) -> RetrievalResult:
    with stages("fit retrieval spaces", 3) as step:
        step("name char TF-IDF")
        name_q, name_d = TfidfSpace("char_wb", (2, 4)).fit_transform(name_text(s1), name_text(s23))
        step("address char TF-IDF")
        addr_q, addr_d = TfidfSpace("char_wb", (3, 4)).fit_transform(
            s1["addr_core"].tolist(), s23["addr_core"].tolist()
        )
        step("BM25")
        bm_q, bm_d = BM25(cfg.bm25_k1, cfg.bm25_b).fit_transform(bm25_text(s1), bm25_text(s23))
    channels = {
        "name_char": (name_q, name_d, cfg.k_name),
        "addr_char": (addr_q, addr_d, cfg.k_addr),
        "bm25": (bm_q, bm_d, cfg.k_bm25),
    }
    source = s23["source"].to_numpy()
    rows: list[pd.DataFrame] = []
    sources = sorted(set(source))
    for src, (ch, (q, d, k)) in progress(
        [(src, item) for src in sources for item in channels.items()], "retrieval channels", len(sources) * 3, "channel"
    ):
        src_rows = np.flatnonzero(source == src).astype(np.int64)
        idx, sc = sparse_topk(q, d[src_rows], k)
        n_q, kk = idx.shape
        ok = sc.ravel() > 0
        rows.append(
            pd.DataFrame(
                {
                    "s1_idx": np.repeat(np.arange(n_q), kk)[ok],
                    "c_idx": src_rows[idx.ravel()][ok],
                    "channel": ch,
                    "rank": np.tile(np.arange(1, kk + 1), n_q)[ok],
                    "score": sc.ravel()[ok],
                }
            )
        )
    for src in sources:
        src_rows = np.flatnonzero(source == src).astype(np.int64)
        pi, pj, pr = _postal_channel(s1, s23, src_rows, name_q, name_d, cfg.k_postal)
        rows.append(
            pd.DataFrame(
                {
                    "s1_idx": pi,
                    "c_idx": pj,
                    "channel": "postal",
                    "rank": pr,
                    "score": np.ones(len(pi), dtype=np.float32),
                }
            )
        )

    hits = pd.concat(rows, ignore_index=True)
    hits["rrf"] = 1.0 / (cfg.rrf_k + hits["rank"])
    wide_rank = hits.pivot_table(index=["s1_idx", "c_idx"], columns="channel", values="rank", aggfunc="min")
    wide_score = hits.pivot_table(index=["s1_idx", "c_idx"], columns="channel", values="score", aggfunc="max")
    fused = hits.groupby(["s1_idx", "c_idx"], sort=True)["rrf"].sum()
    wide = pd.DataFrame(index=fused.index)
    for ch in CHANNELS:
        wide[f"rank_{ch}"] = wide_rank[ch] if ch in wide_rank else np.nan
        if ch != "postal":
            wide[f"ret_{ch}"] = wide_score[ch] if ch in wide_score else np.nan
    wide["rrf"] = fused
    wide["n_channels"] = wide[[f"rank_{c}" for c in CHANNELS]].notna().sum(axis=1)
    wide = wide.reset_index()
    wide["is_s3"] = (source[wide["c_idx"].to_numpy()] == "S3").astype(np.int64)

    if cfg.k_final > 0:
        wide = wide.sort_values(["s1_idx", "is_s3", "rrf", "c_idx"], ascending=[True, True, False, True])
        wide["rank_fused"] = wide.groupby(["s1_idx", "is_s3"]).cumcount() + 1
        wide = wide[wide["rank_fused"] <= cfg.k_final]
    else:
        wide["rank_fused"] = wide.groupby(["s1_idx", "is_s3"])["rrf"].rank(ascending=False, method="first")
    wide = wide.sort_values(["s1_idx", "c_idx"]).reset_index(drop=True)

    kmax = {"name_char": cfg.k_name, "addr_char": cfg.k_addr, "bm25": cfg.k_bm25, "postal": cfg.k_postal}
    for ch in CHANNELS:
        wide[f"rank_{ch}"] = wide[f"rank_{ch}"].fillna(kmax[ch] + 1)
    feats = wide.drop(columns=["s1_idx", "c_idx"]).astype(np.float32)
    return RetrievalResult(
        s1_idx=wide["s1_idx"].to_numpy(np.int64),
        c_idx=wide["c_idx"].to_numpy(np.int64),
        feats=feats.reset_index(drop=True),
        spaces={"name_char": (name_q, name_d), "addr_char": (addr_q, addr_d), "bm25": (bm_q, bm_d)},
    )

In [ ]:
%%writefile er_src/method2_tfidf_retrieval/scorer.py
"""Method 2 pair features and weighted similarity scorer.

    name_score    = mean(name char-TF-IDF cos, name word-TF-IDF cos, token_sort, Jaro-Winkler)
    address_score = mean(addr char-TF-IDF cos, addr word-TF-IDF cos, token_set)
    score         = w_name * name_score + (1 - w_name) * address_score
                    + postal / house-number bonus-penalty - country-conflict penalty

TF-IDF weighting is what separates this from Method 1: generic tokens ("hospital", "traders",
"pvt") carry little weight, distinctive ones ("balaji", "dupont") carry most of it.
A missing address contributes a tuned neutral value; weights, threshold ``tau``, margin ``delta``
and the exclusivity switch are tuned on entity-level F0.5.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any

import numpy as np
import numpy.typing as npt
import pandas as pd

from er_common.decision import exclusive_mask, threshold_margin
from er_common.pairs import PairTable
from er_common.progress import stages
from er_common.similarity import TfidfSpace, TokenSets, cosine_pairs, paired, tri_state
from er_common.tuning import Dim
from method2_tfidf_retrieval.retrieval import RetrievalResult

FloatArr = npt.NDArray[np.float64]
BoolArr = npt.NDArray[np.bool_]


def _col(df: pd.DataFrame, name: str, idx: npt.NDArray[np.int64]) -> npt.NDArray[Any]:
    return df[name].to_numpy()[idx]


def tfidf_features(pt: PairTable, ret: RetrievalResult) -> pd.DataFrame:
    a, b, ia, ib = pt.s1, pt.s23, pt.s1_idx, pt.c_idx
    nq, nd = ret.spaces["name_char"]
    aq, ad = ret.spaces["addr_char"]
    f: dict[str, npt.NDArray[np.float32]] = {}
    with stages(f"M2 features ({len(ia):,} pairs)", 5) as step:
        step("word TF-IDF spaces")
        wn_q, wn_d = TfidfSpace("word", (1, 1)).fit_transform(a["name_core"].tolist(), b["name_core"].tolist())
        wa_q, wa_d = TfidfSpace("word", (1, 1)).fit_transform(a["addr_core"].tolist(), b["addr_core"].tolist())
        step("name TF-IDF / fuzzy")
        na, nb = _col(a, "name_core", ia), _col(b, "name_core", ib)
        f["name_char_cos"] = cosine_pairs(nq, nd, ia, ib)
        f["name_word_cos"] = cosine_pairs(wn_q, wn_d, ia, ib)
        f["name_tsort"] = paired(na, nb, "tsort")
        f["name_jw"] = paired(na, nb, "jw")
        step("address TF-IDF / fuzzy")
        aa, ab = _col(a, "addr_core", ia), _col(b, "addr_core", ib)
        f["addr_char_cos"] = cosine_pairs(aq, ad, ia, ib)
        f["addr_word_cos"] = cosine_pairs(wa_q, wa_d, ia, ib)
        f["addr_tset"] = paired(aa, ab, "tset")
        step("numbers")
        nums = TokenSets(a["nums"].tolist(), b["nums"].tolist()).features(ia, ib)
        step("postal / house / country")
        f["postal"] = tri_state(_col(a, "postal", ia), _col(b, "postal", ib))
        f["house"] = tri_state(_col(a, "house", ia), _col(b, "house", ib))
        f["nums_jaccard"] = nums["jaccard"]
        f["country"] = tri_state(_col(a, "country_norm", ia), _col(b, "country_norm", ib))
    df = pd.DataFrame(f)
    with np.errstate(all="ignore"):
        name_cols = ["name_char_cos", "name_word_cos", "name_tsort", "name_jw"]
        addr_cols = ["addr_char_cos", "addr_word_cos", "addr_tset"]
        df["name_score"] = df[name_cols].mean(axis=1, skipna=True).fillna(0.0)
        df["addr_score"] = df[addr_cols].mean(axis=1, skipna=True)
    return df


@dataclass(frozen=True)
class ScoreParams:
    w_name: float = 0.5
    v_addr_missing: float = 0.5
    b_postal: float = 0.05
    p_postal: float = 0.10
    b_house: float = 0.05
    p_house: float = 0.10
    p_country: float = 0.30
    tau: float = 0.70
    delta: float = 0.15
    exclusive: float = 1.0

    def as_dict(self) -> dict[str, float]:
        return asdict(self)


SPACE: dict[str, Dim] = {
    "v_addr_missing": Dim(0.0, 1.0),
    "w_name": Dim(0.2, 0.8),
    "b_postal": Dim(0.0, 0.2),
    "p_postal": Dim(0.0, 0.3),
    "b_house": Dim(0.0, 0.2),
    "p_house": Dim(0.0, 0.3),
    "p_country": Dim(0.0, 1.0),
    "tau": Dim(0.3, 1.1),
    "delta": Dim(0.0, 0.5),
    "exclusive": Dim(0.0, 1.0, integer=True),
}


class Scorer:
    def __init__(self, pt: PairTable):
        f = pt.feats
        self.name = f["name_score"].to_numpy(np.float64)
        self.addr = f["addr_score"].to_numpy(np.float64)
        self.postal = f["postal"].to_numpy(np.float64)
        self.house = f["house"].to_numpy(np.float64)
        self.country = f["country"].to_numpy(np.float64)
        self.group, self.cand, self.n_groups = pt.s1_idx, pt.c_idx, pt.n_s1

    def score(self, p: ScoreParams) -> FloatArr:
        addr = np.where(np.isnan(self.addr), p.v_addr_missing, self.addr)
        out: FloatArr = (
            p.w_name * self.name
            + (1 - p.w_name) * addr
            + p.b_postal * (self.postal == 1)
            - p.p_postal * (self.postal == 0)
            + p.b_house * (self.house == 1)
            - p.p_house * (self.house == 0)
            - p.p_country * (self.country == 0)
        )
        return out

    def decide(self, p: ScoreParams) -> tuple[BoolArr, FloatArr]:
        s = self.score(p)
        base = s >= p.tau
        if p.exclusive >= 0.5:
            base &= exclusive_mask(self.cand, s, eligible=base)
        return threshold_margin(self.group, s, self.n_groups, tau=p.tau, delta=p.delta, base=base), s

In [ ]:
%%writefile er_src/method2_tfidf_retrieval/method.py
"""Method 2 -- TF-IDF / BM25 retrieval + weighted similarity scoring."""

from __future__ import annotations

import dataclasses
import logging
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import numpy.typing as npt
import pandas as pd

from er_common.io import SplitData
from er_common.metrics import GroupedEvaluator
from er_common.normalize import normalize_records
from er_common.pairs import PairTable
from er_common.tuning import search
from method2_tfidf_retrieval.retrieval import RetrievalConfig, RetrievalResult, retrieve
from method2_tfidf_retrieval.scorer import SPACE, ScoreParams, Scorer, tfidf_features

BoolArr = npt.NDArray[np.bool_]
FloatArr = npt.NDArray[np.float64]

log = logging.getLogger("er.method2")


@dataclass(frozen=True)
class Method2Config:
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    n_random: int = 600
    n_rounds: int = 3
    seed: int = 42


def build_retrieval_pairs(data: SplitData, cfg: RetrievalConfig) -> tuple[PairTable, RetrievalResult]:
    """Normalise + retrieve; shared with Method 3."""
    if isinstance(cfg, dict):  # JSON override
        cfg = RetrievalConfig(**cfg)
    s1 = normalize_records(data.s1)
    s23 = normalize_records(data.s23)
    ret = retrieve(s1, s23, cfg)
    pt = PairTable(s1=s1, s23=s23, s1_idx=ret.s1_idx, c_idx=ret.c_idx)
    return pt, ret


class TfidfRetrievalMethod:
    name = "method2_tfidf_retrieval"

    def __init__(self, config: Method2Config | None = None):
        self.config = config or Method2Config()

    def build_pairs(self, data: SplitData) -> PairTable:
        pt, ret = build_retrieval_pairs(data, self.config.retrieval)
        pt.feats = pd.concat([ret.feats, tfidf_features(pt, ret)], axis=1)
        return pt

    def fit(self, pt: PairTable, gt: Mapping[str, Sequence[str]], active_s1: BoolArr) -> ScoreParams:
        scorer = Scorer(pt)
        evaluator = GroupedEvaluator(pt.s1_idx, pt.labels(gt), pt.n_true(gt), active_s1)

        def objective(p: dict[str, float]) -> float:
            mask, _ = scorer.decide(ScoreParams(**p))
            return evaluator.score(mask)

        best, score = search(
            objective,
            SPACE,
            ScoreParams().as_dict(),
            n_random=self.config.n_random,
            n_rounds=self.config.n_rounds,
            seed=self.config.seed,
        )
        log.info("tuned scorer: in-sample F0.5 %.4f", score)
        return ScoreParams(**best)

    def predict(self, model: ScoreParams, pt: PairTable) -> tuple[BoolArr, FloatArr]:
        return Scorer(pt).decide(model)

    def model_summary(self, model: ScoreParams) -> dict[str, Any]:
        return {k: round(v, 4) for k, v in dataclasses.asdict(model).items()}

In [ ]:
%%writefile er_src/method3_gbdt/__init__.py
"""Method 3: supervised pairwise GBDT + retrieval blocking + hard negatives + F0.5 decision layer."""

In [ ]:
%%writefile er_src/method3_gbdt/features.py
"""Method 3 pair features (~70 columns), all language- and country-agnostic.

Groups
  retrieval   per-channel ranks / scores, RRF, #channels, fused rank, source flag (from Method 2)
  name        TF-IDF char/word cosine, 6 fuzzy scorers, skeleton ratio, IDF-weighted Jaccard,
              containment, exact flags, acronym, trade-name (dba), legal-form agreement,
              first/last-token agreement, length deltas
  address     TF-IDF char/word cosine, fuzzy scorers on core and full address, landmark
              similarity, IDF-weighted Jaccard, postal equality / prefix, house number,
              numeric-token overlap and conflicts
  frequency   how common the name is on each side (chains: many branches share a name)
  context     per-S1: rank / gap of this candidate vs the S1's best; per-candidate: how many S1
              records compete for it and the gap to the best competing S1 (reverse rank)
There is deliberately no country one-hot: the only country signal is label agreement, so an
unseen country (France in test) is handled by the same features.
"""

from __future__ import annotations

from typing import Any

import numpy as np
import numpy.typing as npt
import pandas as pd

from er_common.pairs import PairTable
from er_common.progress import stages
from er_common.similarity import TokenSets, acronym_match, paired, prefix_match, tri_state
from method2_tfidf_retrieval.retrieval import RetrievalResult
from method2_tfidf_retrieval.scorer import tfidf_features

IntArr = npt.NDArray[np.int64]
F32 = npt.NDArray[np.float32]

# features whose partial dependence must be non-decreasing in P(match)
MONOTONE_INCREASING = (
    "name_char_cos",
    "name_word_cos",
    "name_tsort",
    "name_tset",
    "name_jw",
    "name_ratio",
    "name_wjaccard",
    "addr_char_cos",
    "addr_word_cos",
    "addr_tset",
    "addr_tsort",
    "addr_wjaccard",
    "postal",
    "house",
    "country",
)


def _col(df: pd.DataFrame, name: str, idx: IntArr) -> npt.NDArray[Any]:
    return df[name].to_numpy()[idx]


def _eq(a: npt.NDArray[Any], b: npt.NDArray[Any]) -> F32:
    out = np.fromiter((x == y for x, y in zip(a, b, strict=True)), dtype=np.float32, count=len(a))
    empty = np.fromiter((not x or not y for x, y in zip(a, b, strict=True)), dtype=bool, count=len(a))
    out[empty] = np.nan
    return out


def _token_pos_eq(a: npt.NDArray[Any], b: npt.NDArray[Any], pos: int) -> F32:
    out = np.full(len(a), np.nan, dtype=np.float32)
    for i, (x, y) in enumerate(zip(a, b, strict=True)):
        tx, ty = x.split(), y.split()
        if tx and ty:
            out[i] = float(tx[pos] == ty[pos])
    return out


def name_features(pt: PairTable) -> dict[str, F32]:
    a, b, ia, ib = pt.s1, pt.s23, pt.s1_idx, pt.c_idx
    na, nb = _col(a, "name_core", ia), _col(b, "name_core", ib)
    alt_a, alt_b = _col(a, "name_alt", ia), _col(b, "name_alt", ib)
    toks = TokenSets(a["name_core"].tolist(), b["name_core"].tolist()).features(ia, ib)
    alt = np.fmax(np.fmax(paired(alt_a, nb, "tsort"), paired(na, alt_b, "tsort")), paired(alt_a, alt_b, "tsort"))
    len_a = np.fromiter((len(x) for x in na), np.float32, len(na))
    len_b = np.fromiter((len(x) for x in nb), np.float32, len(nb))
    ntok_a = np.fromiter((len(x.split()) for x in na), np.float32, len(na))
    ntok_b = np.fromiter((len(x.split()) for x in nb), np.float32, len(nb))
    return {
        "name_ratio": paired(na, nb, "ratio"),
        "name_partial": paired(na, nb, "partial"),
        "name_tset": paired(na, nb, "tset"),
        "name_lev": paired(na, nb, "lev"),
        "name_norm_tsort": paired(_col(a, "name_norm", ia), _col(b, "name_norm", ib), "tsort"),
        "name_skel_ratio": paired(_col(a, "name_skel", ia), _col(b, "name_skel", ib), "ratio"),
        "name_skel_tset": paired(_col(a, "name_skel", ia), _col(b, "name_skel", ib), "tset"),
        "name_core_exact": _eq(na, nb),
        "name_skel_exact": _eq(_col(a, "name_skel", ia), _col(b, "name_skel", ib)),
        "name_acronym": acronym_match(na, _col(a, "name_acronym", ia), nb, _col(b, "name_acronym", ib)),
        "name_alt_best": alt,
        "name_legal_eq": tri_state(_col(a, "name_legal", ia), _col(b, "name_legal", ib)),
        "name_first_tok_eq": _token_pos_eq(na, nb, 0),
        "name_last_tok_eq": _token_pos_eq(na, nb, -1),
        "name_jaccard": toks["jaccard"],
        "name_wjaccard": toks["wjaccard"],
        "name_containment": toks["containment"],
        "name_only_s1": toks["only_left"],
        "name_only_cand": toks["only_right"],
        "name_len_diff": np.abs(len_a - len_b),
        "name_ntok_diff": np.abs(ntok_a - ntok_b),
        "name_len_min": np.minimum(len_a, len_b),
    }


def address_features(pt: PairTable) -> dict[str, F32]:
    a, b, ia, ib = pt.s1, pt.s23, pt.s1_idx, pt.c_idx
    aa, ab = _col(a, "addr_core", ia), _col(b, "addr_core", ib)
    toks = TokenSets(a["addr_core"].tolist(), b["addr_core"].tolist()).features(ia, ib)
    nums = TokenSets(a["nums"].tolist(), b["nums"].tolist()).features(ia, ib)
    len_a = np.fromiter((len(x.split()) for x in aa), np.float32, len(aa))
    len_b = np.fromiter((len(x.split()) for x in ab), np.float32, len(ab))
    return {
        "addr_tsort": paired(aa, ab, "tsort"),
        "addr_ratio": paired(aa, ab, "ratio"),
        "addr_partial": paired(aa, ab, "partial"),
        "addr_jw": paired(aa, ab, "jw"),
        "addr_full_tset": paired(_col(a, "addr_norm", ia), _col(b, "addr_norm", ib), "tset"),
        "landmark_tset": paired(_col(a, "addr_landmark", ia), _col(b, "addr_landmark", ib), "tset"),
        "addr_jaccard": toks["jaccard"],
        "addr_wjaccard": toks["wjaccard"],
        "addr_containment": toks["containment"],
        "addr_only_s1": toks["only_left"],
        "addr_only_cand": toks["only_right"],
        "addr_ntok_s1": len_a,
        "addr_ntok_cand": len_b,
        "postal_prefix3": prefix_match(_col(a, "postal", ia), _col(b, "postal", ib), 3),
        "nums_containment": nums["containment"],
        "nums_common": nums["common"],
        "nums_only_s1": nums["only_left"],
        "nums_only_cand": nums["only_right"],
    }


def frequency_features(pt: PairTable) -> dict[str, F32]:
    s1_counts = pt.s1["name_core"].value_counts()
    c_counts = pt.s23["name_core"].value_counts()
    n1 = pt.s1["name_core"].map(s1_counts).to_numpy(np.float32)
    n23_of_s1 = pt.s1["name_core"].map(c_counts).fillna(0).to_numpy(np.float32)
    nc = pt.s23["name_core"].map(c_counts).to_numpy(np.float32)
    return {
        "name_freq_s1": np.log1p(n1[pt.s1_idx]),
        "name_freq_s1_in_cands": np.log1p(n23_of_s1[pt.s1_idx]),
        "name_freq_cand": np.log1p(nc[pt.c_idx]),
    }


def _group_top2(g: IntArr, v: npt.NDArray[np.float64]) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
    """Per-pair (max, second max) of ``v`` within its group ``g`` (second = -inf if singleton group)."""
    order = np.lexsort((-v, g))
    gs, vs = g[order], v[order]
    first = np.r_[True, gs[1:] != gs[:-1]]
    starts = np.flatnonzero(first)
    sizes = np.diff(np.r_[starts, len(gs)])
    top = vs[starts]
    second = np.where(sizes > 1, vs[np.minimum(starts + 1, len(vs) - 1)], -np.inf)
    grp_id = np.cumsum(first) - 1
    out_top, out_second = np.empty(len(v)), np.empty(len(v))
    out_top[order] = top[grp_id]
    out_second[order] = second[grp_id]
    return out_top, out_second


def context_features(pt: PairTable, df: pd.DataFrame) -> dict[str, F32]:
    name = df["name_score"].to_numpy(np.float64)
    addr = df["addr_score"].to_numpy(np.float64)
    combo = 0.5 * name + 0.5 * np.nan_to_num(addr, nan=0.5)
    g, c = pt.s1_idx, pt.c_idx
    s1_max, s1_second = _group_top2(g, combo)
    c_max, c_second = _group_top2(c, combo)
    best_other_cand = np.where(combo >= s1_max, s1_second, s1_max)
    best_other_s1 = np.where(combo >= c_max, c_second, c_max)
    n_strong_name = np.bincount(g, weights=(name >= 0.9), minlength=pt.n_s1)
    n_strong_addr = np.bincount(g, weights=(np.nan_to_num(addr) >= 0.9), minlength=pt.n_s1)
    n_comp = np.bincount(c, minlength=len(pt.s23))
    rank_s1 = pd.Series(combo).groupby(g).rank(ascending=False, method="min").to_numpy()
    rank_c = pd.Series(combo).groupby(c).rank(ascending=False, method="min").to_numpy()

    def clip(x: npt.NDArray[np.float64]) -> F32:  # gaps are -inf when there is no competitor
        return np.clip(x, -2.0, 2.0).astype(np.float32)

    return {
        "combo": combo.astype(np.float32),
        "ctx_rank_in_s1": rank_s1.astype(np.float32),
        "ctx_gap_to_s1_best": clip(combo - s1_max),
        "ctx_margin_over_other_cand": clip(combo - best_other_cand),
        "ctx_n_strong_names": n_strong_name[g].astype(np.float32),
        "ctx_n_strong_addrs": n_strong_addr[g].astype(np.float32),
        "ctx_rank_in_cand": rank_c.astype(np.float32),
        "ctx_n_s1_competing": n_comp[c].astype(np.float32),
        "ctx_margin_over_other_s1": clip(combo - best_other_s1),
    }


def build_features(pt: PairTable, ret: RetrievalResult) -> pd.DataFrame:
    base = tfidf_features(pt, ret)
    df = pd.concat([ret.feats.reset_index(drop=True), base], axis=1)
    with stages(f"M3 extra features ({pt.n_pairs:,} pairs)", 4) as step:
        step("name")
        blocks = [name_features(pt)]
        step("address")
        blocks.append(address_features(pt))
        step("name frequency")
        blocks.append(frequency_features(pt))
        for block in blocks:
            for k, v in block.items():
                df[k] = v
        df["name_x_addr"] = (df["name_score"] * df["addr_score"]).astype(np.float32)
        df["name_min_addr"] = np.fmin(df["name_score"], df["addr_score"]).astype(np.float32)
        df["addr_missing"] = df["addr_score"].isna().astype(np.float32)
        step("competition context")
        for k, v in context_features(pt, df).items():
            df[k] = v
    return df.astype(np.float32)

In [ ]:
%%writefile er_src/method3_gbdt/model.py
"""LightGBM training with S1-grouped early stopping and deterministic settings."""

from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass

import lightgbm as lgb
import numpy as np
import numpy.typing as npt
import pandas as pd

from er_common.progress import bar
from method3_gbdt.features import MONOTONE_INCREASING

IntArr = npt.NDArray[np.int64]
BoolArr = npt.NDArray[np.bool_]
FloatArr = npt.NDArray[np.float64]


@dataclass(frozen=True)
class GbdtParams:
    learning_rate: float = 0.03
    num_leaves: int = 31
    min_child_samples: int = 20
    feature_fraction: float = 0.8
    bagging_fraction: float = 0.8
    bagging_freq: int = 1
    lambda_l2: float = 1.0
    max_rounds: int = 3000
    early_stopping: int = 150
    es_fraction: float = 0.15
    monotone: bool = True
    num_threads: int = 4


def lgb_params(p: GbdtParams, seed: int, features: Sequence[str]) -> dict[str, object]:
    params: dict[str, object] = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": p.learning_rate,
        "num_leaves": p.num_leaves,
        "min_child_samples": p.min_child_samples,
        "feature_fraction": p.feature_fraction,
        "bagging_fraction": p.bagging_fraction,
        "bagging_freq": p.bagging_freq,
        "lambda_l2": p.lambda_l2,
        "num_threads": p.num_threads,
        "seed": seed,
        "deterministic": True,
        "force_row_wise": True,
        "verbosity": -1,
    }
    if p.monotone:
        params["monotone_constraints"] = [1 if f in MONOTONE_INCREASING else 0 for f in features]
        params["monotone_constraints_method"] = "advanced"
    return params


def _es_split(groups: IntArr, fraction: float, seed: int) -> BoolArr:
    """Boolean mask of rows whose S1 group is held out for early stopping."""
    uniq = np.unique(groups)
    rng = np.random.default_rng(seed)
    held = rng.choice(uniq, size=max(1, round(fraction * len(uniq))), replace=False)
    return np.isin(groups, held)


def _round_bar(desc: str, total: int) -> tuple[object, object]:
    """LightGBM callback that advances a progress bar once per boosting round."""
    pb = bar(desc, total, "round")

    def cb(env: lgb.callback.CallbackEnv) -> None:
        pb.update(1)
        if env.evaluation_result_list:
            _, metric, value, _ = env.evaluation_result_list[0][:4]
            pb.set_postfix_str(f"{metric}={value:.5f}", refresh=False)

    return pb, cb


def train(
    x: pd.DataFrame,
    y: BoolArr,
    groups: IntArr,
    p: GbdtParams,
    seed: int,
    n_rounds: int | None = None,
) -> tuple[lgb.Booster, int]:
    """With ``n_rounds=None``: early-stop on a held-out slice of S1 groups, return (model, best_iter).
    Otherwise train on everything for exactly ``n_rounds``."""
    features = list(x.columns)
    params = lgb_params(p, seed, features)
    if n_rounds is None:
        es = _es_split(groups, p.es_fraction, seed)
        dtrain = lgb.Dataset(x.loc[~es], label=y[~es].astype(int), feature_name=features, free_raw_data=True)
        dvalid = lgb.Dataset(x.loc[es], label=y[es].astype(int), reference=dtrain)
        pb, cb = _round_bar("LightGBM (early stopping)", p.max_rounds)
        with pb:  # type: ignore[attr-defined]
            booster = lgb.train(
                params,
                dtrain,
                num_boost_round=p.max_rounds,
                valid_sets=[dvalid],
                callbacks=[lgb.early_stopping(p.early_stopping, verbose=False), cb],  # type: ignore[list-item]
            )
        return booster, int(booster.best_iteration or p.max_rounds)
    dtrain = lgb.Dataset(x, label=y.astype(int), feature_name=features, free_raw_data=True)
    pb, cb = _round_bar("LightGBM (final fit)", max(1, n_rounds))
    with pb:  # type: ignore[attr-defined]
        booster = lgb.train(params, dtrain, num_boost_round=max(1, n_rounds), callbacks=[cb])  # type: ignore[list-item]
    return booster, n_rounds


def predict(booster: lgb.Booster, x: pd.DataFrame) -> FloatArr:
    it = booster.best_iteration if booster.best_iteration and booster.best_iteration > 0 else None
    return np.asarray(booster.predict(x, num_iteration=it), dtype=np.float64)

In [ ]:
%%writefile er_src/method3_gbdt/policy.py
"""Decision policy on top of calibrated pair probabilities.

    raw P(match)  --Platt-->  calibrated p  --exclusivity-->  q  --strategy-->  match set

Exclusivity (an S2/S3 record belongs to at most one S1 entity):
    none  q = p
    hard  q = p for the S1 with the highest p for that record, 0 for the others
    soft  treat ownership of a record as one categorical choice among the S1 records competing
          for it plus "no owner":  q_i = o_i / (1 + sum_j o_j),  o = p / (1 - p).
          A record claimed by one S1 keeps q = p; two strong claims (0.9, 0.9) both drop to 0.47,
          i.e. ambiguous records are withheld -- exactly what a precision-weighted metric wants.

Strategy:
    expected_f  per S1, predict the top-k maximising E[F0.5] (Bayes-optimal for independent,
                calibrated labels; handles singletons and multi-matches without a threshold)
    threshold   q >= tau and q >= best_q_of_S1 - delta

Every combination is scored on out-of-fold probabilities; the best is kept.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, replace

import numpy as np
import numpy.typing as npt

from er_common.decision import PlattCalibrator, exclusive_mask, expected_fbeta_topk, threshold_margin
from er_common.metrics import GroupedEvaluator
from er_common.pairs import PairTable
from er_common.progress import progress

FloatArr = npt.NDArray[np.float64]
BoolArr = npt.NDArray[np.bool_]
IntArr = npt.NDArray[np.int64]

EXCLUSIVITY = ("none", "hard", "soft")
STRATEGIES = ("expected_f", "threshold")
TAUS = tuple(np.round(np.arange(0.05, 0.96, 0.05), 2).tolist())
DELTAS = (0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0)


@dataclass(frozen=True)
class Policy:
    calib_a: float = 1.0
    calib_b: float = 0.0
    exclusivity: str = "soft"
    strategy: str = "expected_f"
    tau: float = 0.5
    delta: float = 1.0

    def as_dict(self) -> dict[str, object]:
        return asdict(self)


def apply_exclusivity(p: FloatArr, cand: IntArr, mode: str) -> FloatArr:
    if mode == "none":
        return p
    if mode == "hard":
        return np.where(exclusive_mask(cand, p), p, 0.0)
    if mode == "soft":
        o = np.clip(p, 0.0, 1 - 1e-6)
        o = o / (1.0 - o)
        total = np.bincount(cand, weights=o, minlength=int(cand.max()) + 1 if len(cand) else 0)
        return o / (1.0 + total[cand])
    raise ValueError(f"unknown exclusivity mode {mode!r}")


def decide(policy: Policy, q: FloatArr, pt: PairTable) -> BoolArr:
    if policy.strategy == "expected_f":
        return expected_fbeta_topk(pt.s1_idx, q, pt.n_s1)
    if policy.strategy == "threshold":
        return threshold_margin(pt.s1_idx, q, pt.n_s1, policy.tau, policy.delta)
    raise ValueError(f"unknown strategy {policy.strategy!r}")


def apply_policy(policy: Policy, raw: FloatArr, pt: PairTable) -> tuple[BoolArr, FloatArr]:
    p = PlattCalibrator(policy.calib_a, policy.calib_b)(raw)
    q = apply_exclusivity(p, pt.c_idx, policy.exclusivity)
    return decide(policy, q, pt), q


def fit_policy(
    raw: FloatArr,
    pt: PairTable,
    label: BoolArr,
    n_true: IntArr,
    active_s1: BoolArr,
    exclusivity: tuple[str, ...] = EXCLUSIVITY,
    strategies: tuple[str, ...] = STRATEGIES,
    fixed_threshold: float | None = None,
) -> tuple[Policy, float]:
    """Fit calibration on active S1 pairs and pick the best (exclusivity, strategy, tau, delta)."""
    active_pairs = active_s1[pt.s1_idx]
    cal = PlattCalibrator.fit(raw[active_pairs], label[active_pairs])
    p = cal(raw)
    ev = GroupedEvaluator(pt.s1_idx, label, n_true, active_s1)
    best: tuple[float, Policy] | None = None

    def consider(score: float, pol: Policy) -> None:
        nonlocal best
        if best is None or score > best[0] + 1e-9:
            best = (score, pol)

    for excl in progress(exclusivity, "fit decision policy", len(exclusivity), "mode"):
        q = apply_exclusivity(p, pt.c_idx, excl)
        base = Policy(calib_a=cal.a, calib_b=cal.b, exclusivity=excl)
        if "expected_f" in strategies:
            pol = replace(base, strategy="expected_f")
            consider(ev.score(decide(pol, q, pt)), pol)
        if "threshold" in strategies:
            grid = [(fixed_threshold, 1.0)] if fixed_threshold is not None else [(t, d) for t in TAUS for d in DELTAS]
            for tau, delta in grid:
                pol = replace(base, strategy="threshold", tau=tau, delta=delta)
                consider(ev.score(threshold_margin(pt.s1_idx, q, pt.n_s1, tau, delta)), pol)
    assert best is not None
    return best[1], best[0]

In [ ]:
%%writefile er_src/method3_gbdt/method.py
"""Method 3 -- supervised pairwise GBDT on retrieval candidates (hard negatives by construction).

Training pairs are exactly the candidates Method 2's retrieval produces for the training S1
records, labelled with the ground truth. Every negative therefore already looks like a match to
at least one retrieval channel (same-name branches, same-building neighbours, shared postal
codes, near-identical names) -- the hard negatives the model must learn to reject -- and train
and inference see the same pair distribution, so probabilities stay calibrated.

Validation: K-fold over S1 entities -> out-of-fold P(match) for every train pair; the decision
policy (calibration, exclusivity, strategy, thresholds) is then fitted on K-1 folds' OOF
probabilities and scored on the remaining fold (nested), so the reported F0.5 is out-of-sample
for both the model and the decision layer.
"""

from __future__ import annotations

import logging
import time
from collections.abc import Mapping, Sequence
from dataclasses import asdict, dataclass, field
from typing import Any

import lightgbm as lgb
import numpy as np
import numpy.typing as npt

from er_common.folds import stratified_s1_folds
from er_common.io import SplitData
from er_common.metrics import GroupedEvaluator
from er_common.pairs import PairTable
from er_common.progress import progress
from method2_tfidf_retrieval.method import build_retrieval_pairs
from method2_tfidf_retrieval.retrieval import RetrievalConfig
from method3_gbdt import model as gbdt
from method3_gbdt.features import build_features
from method3_gbdt.model import GbdtParams
from method3_gbdt.policy import Policy, apply_policy, fit_policy

BoolArr = npt.NDArray[np.bool_]
FloatArr = npt.NDArray[np.float64]
IntArr = npt.NDArray[np.int64]
GT = Mapping[str, Sequence[str]]

log = logging.getLogger("er.method3")

# decision-layer variants reported in the ablation table: (exclusivity modes, strategies)
ABLATIONS: dict[str, tuple[tuple[str, ...], tuple[str, ...]]] = {
    "threshold_tuned": (("none",), ("threshold",)),
    "threshold_tuned+exclusivity": (("hard", "soft"), ("threshold",)),
    "expected_f": (("none",), ("expected_f",)),
    "expected_f+soft_exclusivity": (("soft",), ("expected_f",)),
    "auto (selected)": (("none", "hard", "soft"), ("expected_f", "threshold")),
}


@dataclass(frozen=True)
class Method3Config:
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    gbdt: GbdtParams = field(default_factory=GbdtParams)
    inner_folds: int = 5
    loco: bool = True
    seed: int = 42


@dataclass
class FittedGbdt:
    booster: lgb.Booster
    policy: Policy
    features: list[str]
    n_rounds: int
    oof_policy_f05: float
    importance: dict[str, float]


def _folds_for(pt: PairTable, gt: GT, active_s1: BoolArr, n_folds: int, seed: int) -> IntArr:
    """Stratified S1-level fold ids (-1 for inactive S1). Identical to the runner's validation
    folds when all S1 are active, so ``predict`` alone and ``all`` produce the same model."""
    ids = [s for s, a in zip(pt.s1_ids, active_s1, strict=True) if a]
    countries = pt.s1["country_norm"].to_numpy()[active_s1].tolist()
    fmap = stratified_s1_folds(ids, countries, gt, n_folds, seed)
    return np.array([fmap.get(s, -1) for s in pt.s1_ids], dtype=np.int64)


class GbdtMethod:
    name = "method3_gbdt"

    def __init__(self, config: Method3Config | None = None):
        self.config = config or Method3Config()
        self._oof_cache: dict[str, Any] | None = None

    # -------------------------------------------------------------------------------- pairs
    def build_pairs(self, data: SplitData) -> PairTable:
        pt, ret = build_retrieval_pairs(data, self.config.retrieval)
        t0 = time.perf_counter()
        pt.feats = build_features(pt, ret)
        log.info("features: %d pairs x %d columns [%.1fs]", pt.n_pairs, pt.feats.shape[1], time.perf_counter() - t0)
        return pt

    # -------------------------------------------------------------------------------- helpers
    def _params(self) -> GbdtParams:
        p = self.config.gbdt
        return GbdtParams(**p) if isinstance(p, dict) else p

    def _oof(self, pt: PairTable, label: BoolArr, folds: IntArr, n_folds: int) -> tuple[FloatArr, list[int]]:
        oof = np.full(pt.n_pairs, np.nan)
        iters: list[int] = []
        pair_fold = folds[pt.s1_idx]
        for k in progress(range(n_folds), "out-of-fold models", n_folds, "fold"):
            t0 = time.perf_counter()
            tr = (pair_fold != k) & (pair_fold >= 0)
            te = pair_fold == k
            booster, it = gbdt.train(
                pt.feats.loc[tr], label[tr], pt.s1_idx[tr], self._params(), seed=self.config.seed + k
            )
            oof[te] = gbdt.predict(booster, pt.feats.loc[te])
            iters.append(it)
            log.info("  oof fold %d/%d: %d rounds [%.1fs]", k + 1, n_folds, it, time.perf_counter() - t0)
        return oof, iters

    # -------------------------------------------------------------------------------- validation
    def cross_validate(
        self, pt: PairTable, gt: GT, folds: IntArr, n_folds: int
    ) -> tuple[BoolArr, FloatArr, list[dict[str, Any]]]:
        label = pt.labels(gt)
        n_true = pt.n_true(gt)
        oof, iters = self._oof(pt, label, folds, n_folds)
        self._oof_cache = {"pt": id(pt), "folds": folds.tobytes(), "oof": oof, "iters": iters}
        pair_fold = folds[pt.s1_idx]

        info: list[dict[str, Any]] = []
        results: dict[str, float] = {}
        final_pred = np.zeros(pt.n_pairs, dtype=bool)
        final_q = np.zeros(pt.n_pairs)
        for variant, (excl, strat) in progress(ABLATIONS.items(), "decision-layer ablation", len(ABLATIONS), "variant"):
            pred = np.zeros(pt.n_pairs, dtype=bool)
            q_all = np.zeros(pt.n_pairs)
            for k in range(n_folds):
                policy, _ = fit_policy(oof, pt, label, n_true, folds != k, excl, strat)
                mask, q = apply_policy(policy, oof, pt)
                sel = pair_fold == k
                pred[sel], q_all[sel] = mask[sel], q[sel]
                if variant.startswith("auto"):
                    info.append({"fold": k, "rounds": iters[k], "policy": policy.as_dict()})
            results[variant] = GroupedEvaluator(pt.s1_idx, label, n_true, np.ones(pt.n_s1, bool)).score(pred)
            if variant.startswith("auto"):
                final_pred, final_q = pred, q_all
        raw_half = GroupedEvaluator(pt.s1_idx, label, n_true, np.ones(pt.n_s1, bool)).score(oof >= 0.5)
        ablation = {"raw_p>=0.5": round(raw_half, 4), **{k: round(v, 4) for k, v in results.items()}}
        log.info("decision-layer ablation (nested CV F0.5): %s", ablation)
        info.append({"decision_ablation_nested_cv_f05": ablation})

        auc_like = float(np.mean(oof[label])) - float(np.mean(oof[~label])) if label.any() else 0.0
        info.append({"oof_mean_p_pos_minus_neg": round(auc_like, 4), "oof_rounds": iters})
        if self.config.loco:
            info.append({"leave_one_country_out": self._loco(pt, label, n_true, oof, final_pred)})
        return final_pred, final_q, info

    def _loco(self, pt: PairTable, label: BoolArr, n_true: IntArr, oof: FloatArr, cv_pred: BoolArr) -> dict[str, Any]:
        """Train without one country, score it: a proxy for the unseen-country (France) shift."""
        countries = pt.s1["country_norm"].to_numpy()
        out: dict[str, Any] = {}
        uniq = [c for c in sorted(set(countries)) if (countries == c).sum() >= 50]
        if len(uniq) < 2:
            return {"skipped": "fewer than two countries with >= 50 S1 records"}
        for c in progress(uniq, "leave-one-country-out", len(uniq), "country"):
            held = countries == c
            tr = ~held[pt.s1_idx]
            booster, _ = gbdt.train(pt.feats.loc[tr], label[tr], pt.s1_idx[tr], self._params(), seed=self.config.seed)
            p = oof.copy()
            p[~tr] = gbdt.predict(booster, pt.feats.loc[~tr])
            policy, _ = fit_policy(oof, pt, label, n_true, ~held)
            mask, _ = apply_policy(policy, p, pt)
            ev = GroupedEvaluator(pt.s1_idx, label, n_true, held)
            out[c] = {
                "f05_trained_without": round(ev.score(mask), 4),
                "f05_in_distribution_cv": round(ev.score(cv_pred), 4),
            }
            log.info(
                "LOCO %s: F0.5 %.4f trained without it vs %.4f in-distribution CV",
                c,
                out[c]["f05_trained_without"],
                out[c]["f05_in_distribution_cv"],
            )
        return out

    # -------------------------------------------------------------------------------- fit / predict
    def fit(self, pt: PairTable, gt: GT, active_s1: BoolArr) -> FittedGbdt:
        label = pt.labels(gt)
        n_true = pt.n_true(gt)
        folds = _folds_for(pt, gt, active_s1, self.config.inner_folds, self.config.seed)
        cache = self._oof_cache
        if cache is not None and cache["pt"] == id(pt) and cache["folds"] == folds.tobytes():
            oof, iters = cache["oof"], cache["iters"]
            log.info("reusing out-of-fold predictions from validation (same folds)")
        else:
            oof, iters = self._oof(pt, label, folds, self.config.inner_folds)
        policy, score = fit_policy(np.nan_to_num(oof, nan=0.0), pt, label, n_true, active_s1)
        n_rounds = int(np.median(iters) / (1.0 - self._params().es_fraction))
        tr = active_s1[pt.s1_idx]
        booster, _ = gbdt.train(
            pt.feats.loc[tr], label[tr], pt.s1_idx[tr], self._params(), seed=self.config.seed, n_rounds=n_rounds
        )
        gain = booster.feature_importance(importance_type="gain")
        total = float(gain.sum()) or 1.0
        importance = {
            f: round(float(g) / total, 4)
            for f, g in sorted(zip(booster.feature_name(), gain, strict=True), key=lambda x: -x[1])
        }
        log.info("final model: %d rounds; policy %s (OOF F0.5 %.4f)", n_rounds, policy.as_dict(), score)
        return FittedGbdt(booster, policy, list(pt.feats.columns), n_rounds, score, importance)

    def predict(self, model: FittedGbdt, pt: PairTable) -> tuple[BoolArr, FloatArr]:
        missing = [f for f in model.features if f not in pt.feats.columns]
        if missing:
            raise ValueError(f"feature mismatch between train and inference: {missing[:5]}")
        raw = gbdt.predict(model.booster, pt.feats.loc[:, model.features])
        return apply_policy(model.policy, raw, pt)

    def model_summary(self, model: FittedGbdt) -> dict[str, Any]:
        top = dict(list(model.importance.items())[:25])
        return {
            "n_rounds": model.n_rounds,
            "policy": model.policy.as_dict(),
            "oof_policy_f05": round(model.oof_policy_f05, 4),
            "top_feature_gain_share": top,
            "gbdt_params": asdict(self._params()),
        }

## Run

In [ ]:
import logging
import sys

sys.path.insert(0, "er_src")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

from er_common.runner import main
from method3_gbdt.method import GbdtMethod

main(GbdtMethod(), [COMMAND, "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--folds", str(N_FOLDS)])

## Results

In [ ]:
import json

import pandas as pd

report = os.path.join(OUT_DIR, "validation", "report.json")
if os.path.isfile(report):
    v = json.load(open(report))["validation"]
    print("CV F0.5:", round(v["cv"]["f05"], 4), "| per fold:", v["per_fold_f05"])
    print("per country:", {k: x["f05"] for k, x in v["per_country"].items()})
    print("candidate pair recall:", round(v["blocking"]["pair_recall"], 4))

submission = os.path.join(OUT_DIR, "test", "matching_results.tsv")
if os.path.isfile(submission):
    print("submission files:", os.path.join(OUT_DIR, "test"))
    display(pd.read_csv(submission, sep="\t", dtype=str, keep_default_na=False).head(10))